# Structured R1 — Local Temporal Branch Ablation Study

This notebook continues the controlled ablation sequence built on top of **Structured R1**.

The preceding participation-branch ablation showed that the strongest participation representation was:

```text
Participation evidence = `speaks` only
```

while removing `filtered_turns` improved the overall Structured R1 configuration.

The resulting `speaks`-only setup therefore becomes the fixed baseline for the next stage of analysis: **local temporal evidence ablation**.

## Baseline entering this study

The starting configuration retains:

- participant-level `speaks`,
- all local temporal features,
- all global temporal features,
- coarse semantic summaries,
- focused semantic summaries,
- the frozen NORMAL temporal reference,
- the Structured R1 output schema.

The `filtered_turns` field remains removed throughout this notebook.

The participation-selected baseline achieves:

| Configuration | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| **Structured R1 + speaks only** | **78/100** | **90/100** | **89/100** | **100/100** | **89.25%** |

The purpose of this notebook is to determine which components of the **local temporal branch** are actually useful to the unified reasoner.

---

## Local temporal evidence

The local temporal branch contains two conceptually distinct groups of information.

### 1. Response-offset distribution

This group describes the timing of valid conversational handoffs from Participant A to Participant B.

It includes:

- `signed_strict_offsets_seconds`
- `num_signed_strict_offsets`
- `offset_mean_seconds`
- `offset_median_seconds`
- `offset_max_seconds`
- `offset_p75_seconds`
- `offset_p90_seconds`
- `num_offsets_above_1_5_seconds`
- `percent_offsets_above_1_5_seconds`

These fields are treated as a **single conceptual feature group** because they are different summaries of the same underlying quantity: the distribution of response offsets between participant turns.

### 2. Overlap evidence

This group describes simultaneous speaking activity through:

- `clean_overlap_seconds`
- `clean_overlap_percent`

These fields are also treated as one conceptual group because both encode the same underlying phenomenon: the amount of temporal overlap between the two participant speaking timelines.

For this reason, the ablations are performed **group-wise rather than feature-by-feature**. Removing individual statistics from within one group would largely test redundant representations of the same temporal signal rather than genuinely distinct sources of evidence.

---

# Ablation Design

Three controlled local-temporal interventions are evaluated.

## A2 — Remove the entire local temporal branch

All local temporal evidence is removed.

The model therefore retains:

```text
speaks
+
global temporal evidence
+
semantic evidence
```

but receives no local response-offset or overlap information.

The corresponding `local_temporal_assessment` field is also removed from the Structured R1 output schema because there is no local evidence left to assess.

This experiment measures the overall contribution of the local temporal branch.

---

## L1 — Remove overlap evidence

The overlap group is removed:

```text
clean_overlap_seconds
clean_overlap_percent
```

while the complete response-offset distribution is retained.

This produces an **offsets-only local temporal representation**.

The purpose is to test whether the response-offset group contains the useful local timing signal without requiring overlap information.

---

## L2 — Remove response-offset distribution

The complete offset-distribution group is removed together:

```text
signed_strict_offsets_seconds
num_signed_strict_offsets
offset_mean_seconds
offset_median_seconds
offset_max_seconds
offset_p75_seconds
offset_p90_seconds
num_offsets_above_1_5_seconds
percent_offsets_above_1_5_seconds
```

while overlap evidence is retained.

This produces an **overlap-only local temporal representation**.

Because all offset statistics describe the same underlying handoff-delay distribution, they are removed as one coherent evidence group rather than as separate scalar ablations.

---

## Experimental control

Across all three interventions, the non-ablated components remain fixed:

- the same 400-case development set;
- the same Qwen2.5-Omni model;
- the same deterministic decoding settings;
- the same `speaks`-only participation representation;
- the same global temporal branch;
- the same frozen NORMAL reference;
- the same coarse semantic summaries;
- the same focused semantic summaries;
- the same Structured R1 decision policy;
- the same structured final output format.

Each ablation removes both the targeted evidence fields and the corresponding prompt instructions.

The notebook uses manually adapted and explicitly inspected prompts rather than automatic prompt rewriting, ensuring that each intervention is controlled and auditable before the 400-case inference run begins.

## Main results

The resulting local-temporal comparison is:

| Configuration | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| Speaks-only baseline | 78/100 | 90/100 | 89/100 | 100/100 | 89.25% |
| No local temporal branch | 37/100 | 99/100 | 99/100 | 100/100 | 83.75% |
| **Offsets only** | **76/100** | **93/100** | **96/100** | **100/100** | **91.25%** |
| Overlap only | 41/100 | 95/100 | 95/100 | 100/100 | 82.75% |

The ablation shows that the complete local branch is not uniformly beneficial.

Removing all local temporal evidence strongly increases anomaly sensitivity but causes a major loss of NORMAL preservation. More importantly, the two local feature groups behave differently:

- **response offsets** provide the strongest local temporal representation;
- **overlap** is comparatively ambiguous and reduces NORMAL preservation.

The `offsets-only` configuration therefore becomes the selected local-temporal setup for the subsequent ablation stages.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, prompt inspection, evaluation result, and diagnostic output is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install dependencies

Run this cell once in a fresh Colab runtime. Then select:

`Runtime → Restart session`

and continue from the next cell.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 139.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 80.3 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and configure artifact paths

The folder name still contains `1_2_3sec` for historical reasons. The finalized 400-case database is loaded from the exact path below.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Mounted at /content/drive
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the frozen NORMAL reference statistics

Both frozen JSON files contain separate reference profiles. This experiment deliberately extracts and retains only:

```text
reference_profile = NORMAL
```

The lag reference profiles are not included in the binary-only experiment state at this stage.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and audit the final 400-case database

This cell verifies:

- exactly 400 unique cases;
- exactly 100 cases per family;
- 100 `NORMAL` and 300 `ANOMALOUS` binary labels;
- participant-level `speaks` agrees with the final filtered VAD turns;
- every sample has all four coarse and all four focused summaries;
- no focused summary contains the legacy `speaks` field.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Participant-level `speaks` overview

The field is derived exclusively from each sample's final filtered VAD turns.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Interactive case inspection

The inspector supports all four families:

- `NORMAL`
- `WRONG_PARTNER`
- `LAG`
- `SILENT_PARTNER`

It can display participant metadata, VAD-derived `speaks`, filtered turns, temporal features, semantic summaries, and the complete sample JSON.

The direct function can also be used without widgets:

```python
inspect_case(
    family="normal",
    sample_index=0,
)
```

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

## 7. Load Qwen2.5-Omni Thinker


In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


# Ηelper Functions For Qwen Calls

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


# Shared frozen NORMAL reference text

This is the same reference-text construction used by the original binary experiments. No anomaly-specific profile is added here; R2 and R3 load their exact saved prompts, which already contain the frozen LAG₂/LAG₃ profile text.

In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

# Shared full-semantics input and reasoning helpers

The input projection below is copied from the original binary consolidation notebook. It passes exactly the same participation, turn, temporal, coarse-semantic, and focused-semantic fields.

The reasoning schema is categorical rather than free-form so that it can be parsed and inspected consistently.

In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The complete original Structured R1 projection is retained here
# as an internal source projection. The manual local-ablation helper
# creates a deep-copied model-facing view that removes filtered turns
# and retains only participant-level `speaks` for every experiment.
#
# The three source prompts are loaded from their exact saved
# prompt_template.txt files.
#
# The only prompt change is replacement of the original binary
# OUTPUT block with one common structured-reasoning JSON schema.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 256
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key in parsed:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for key in REASONING_SCHEMA_KEYS:

            if normalized[
                key
            ] not in REASONING_ALLOWED_VALUES[
                key
            ]:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        "Do not include reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "The exact original binary OUTPUT block was "
            "replaced by the common structured-reasoning "
            "OUTPUT block."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "binary OUTPUT block -> "
            "structured reasoning OUTPUT block"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 400


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 400


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ] != results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 100


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


# Manual local-temporal ablation helpers

The helper below **does not edit prompts**. It only:

1. loads the already validated A1-T Structured R1 prompt as the speaks-only baseline;
2. prints that original prompt for inspection;
3. removes the requested evidence fields from a deep-copied model-facing payload;
4. accepts a complete manually written prompt template;
5. validates, saves, inspects and executes that exact template.


In [ ]:
# ============================================================
# MANUAL LOCAL-TEMPORAL ABLATION HELPERS
#
# IMPORTANT DESIGN:
#   - NO automatic prompt rewriting is performed.
#   - The validated A1-T prompt (speaks only, no filtered turns)
#     is loaded only as the baseline prompt to print/inspect.
#   - Every ablation prompt is supplied manually as a complete
#     prompt template.
#   - Every model-facing payload contains participant `speaks`
#     only; filtered turn lists are removed in all three runs.
#
# Local ablations:
#   A2 : remove the complete local-temporal branch
#   L1 : remove overlap evidence only
#   L2 : remove offset-distribution evidence only
# ============================================================

from IPython.display import display

R1_SOURCE_EXPERIMENT_NAME = (
    "binary_only_consolidation_"
    "normal_definition_v2"
)

SPEAKS_ONLY_BASELINE_EXPERIMENT_NAME = (
    "ablation_a1_no_turns_keep_speaks"
)

SPEAKS_ONLY_BASELINE_PROMPT_PATH = (
    OUT_DIR
    / SPEAKS_ONLY_BASELINE_EXPERIMENT_NAME
    / "prompt_template.txt"
)

LOCAL_OVERLAP_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",
]

LOCAL_OFFSET_DISTRIBUTION_FIELDS = [
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]

assert (
    LOCAL_OVERLAP_FIELDS
    + LOCAL_OFFSET_DISTRIBUTION_FIELDS
    == LOCAL_FEATURE_FIELDS
)

LOCAL_ABLATION_MODES = {
    "NO_LOCAL_BRANCH",
    "NO_OVERLAP",
    "NO_OFFSET_DISTRIBUTION",
}

STANDARD_REASONING_FIELDS_LOCAL_ABLATION = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]

INSPECTED_MANUAL_LOCAL_ABLATIONS = set()


# ============================================================
# A2 OUTPUT SCHEMA — LOCAL ASSESSMENT REMOVED
# ============================================================

NO_LOCAL_REASONING_OUTPUT_BLOCK = (
    STRUCTURED_REASONING_OUTPUT_BLOCK
    .replace(
        """- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

""",
        "",
    )
    .replace(
        (
            "The local and global temporal assessments must reflect "
            "the reliability\n"
            "rules already defined in the prompt."
        ),
        (
            "The global temporal assessment must reflect the reliability\n"
            "rules already defined in the prompt."
        ),
    )
    .replace(
        (
            '  "local_temporal_assessment": '
            '"NORMAL or ANOMALOUS or LIMITED",\n'
        ),
        "",
    )
)

assert (
    "local_temporal_assessment"
    not in NO_LOCAL_REASONING_OUTPUT_BLOCK
)

FULL_LOCAL_SCHEMA_KEYS = list(
    REASONING_SCHEMA_KEYS
)

FULL_LOCAL_ALLOWED_VALUES = copy.deepcopy(
    REASONING_ALLOWED_VALUES
)

NO_LOCAL_SCHEMA_KEYS = [
    key
    for key in REASONING_SCHEMA_KEYS
    if key != "local_temporal_assessment"
]

NO_LOCAL_ALLOWED_VALUES = copy.deepcopy(
    REASONING_ALLOWED_VALUES
)

NO_LOCAL_ALLOWED_VALUES.pop(
    "local_temporal_assessment"
)


# ============================================================
# LOAD THE VALIDATED SPEAKS-ONLY STRUCTURED R1 BASELINE PROMPT
# ============================================================

def load_speaks_only_baseline_prompt_template():
    assert SPEAKS_ONLY_BASELINE_PROMPT_PATH.exists(), (
        "The validated A1-T speaks-only prompt template was not found:\n"
        f"{SPEAKS_ONLY_BASELINE_PROMPT_PATH}\n\n"
        "Run the A1-T configuration/inspection cell in the participation "
        "ablation notebook first so its prompt_template.txt is saved."
    )

    template = SPEAKS_ONLY_BASELINE_PROMPT_PATH.read_text(
        encoding="utf-8"
    )

    required_markers = [
        "Whether each participant speaks",
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "STRUCTURED REASONING OUTPUT",
        '"participation_assessment"',
        '"local_temporal_assessment"',
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for marker in required_markers:
        assert marker in template, (
            "The saved A1-T baseline prompt is missing the marker: "
            f"{marker}"
        )

    forbidden_markers = [
        "Filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "Use the actual filtered turn lists",
        "Use the speaks fields together with the actual filtered turn lists",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
    ]

    for marker in forbidden_markers:
        assert marker not in template, (
            "Filtered-turn content unexpectedly exists in the saved "
            f"speaks-only baseline prompt: {marker}"
        )

    return template


# ============================================================
# SPEAKS-ONLY FULL MODEL-FACING INPUT
#
# The original database and the shared full payload remain unchanged.
# A deep copy is made and filtered_turns are removed only from the
# model-facing payload used by this notebook.
# ============================================================

def build_speaks_only_full_model_input(case):
    payload = copy.deepcopy(
        build_binary_model_input(case)
    )

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_payload = payload[role]

        assert "speaks" in participant_payload
        assert "filtered_turns" in participant_payload

        participant_payload.pop(
            "filtered_turns"
        )

        assert set(participant_payload) == {
            "speaks"
        }

    return payload


# ============================================================
# LOCAL ABLATION MODEL-FACING INPUT
# ============================================================

def build_manual_local_ablation_model_input(
    case,
    mode,
):
    assert mode in LOCAL_ABLATION_MODES

    payload = build_speaks_only_full_model_input(
        case
    )

    if mode == "NO_LOCAL_BRANCH":
        payload.pop(
            "local_temporal_features"
        )

    elif mode == "NO_OVERLAP":
        local_features = payload[
            "local_temporal_features"
        ]

        for field in LOCAL_OVERLAP_FIELDS:
            local_features.pop(field)

        assert set(local_features) == set(
            LOCAL_OFFSET_DISTRIBUTION_FIELDS
        )

    elif mode == "NO_OFFSET_DISTRIBUTION":
        local_features = payload[
            "local_temporal_features"
        ]

        for field in LOCAL_OFFSET_DISTRIBUTION_FIELDS:
            local_features.pop(field)

        assert set(local_features) == set(
            LOCAL_OVERLAP_FIELDS
        )

    else:
        raise ValueError(
            f"Unknown local ablation mode: {mode}"
        )

    return payload


# ============================================================
# ONE COMMON MANUAL TEMPLATE RENDERER
# ============================================================

def render_manual_prompt_template(
    prompt_template,
    payload,
):
    assert isinstance(prompt_template, str)
    assert prompt_template.strip()

    local_features = payload.get(
        "local_temporal_features"
    )

    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=(
            json.dumps(
                local_features,
                indent=2,
                ensure_ascii=False,
            )
            if local_features is not None
            else ""
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = prompt.lower()

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert forbidden_key.lower() not in prompt_lower, (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )

    for forbidden_turn_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "{participant_a_turns}",
        "{participant_b_turns}",
    ]:
        assert forbidden_turn_marker not in prompt_lower, (
            "Filtered-turn evidence leaked into the model prompt: "
            f"{forbidden_turn_marker}"
        )

    return prompt


# ============================================================
# PRINT THE ORIGINAL SPEAKS-ONLY STRUCTURED R1 PROMPT
#
# Run this BEFORE manually defining each ablation prompt.
# It prints the same validated A1-T baseline prompt with all local,
# global and semantic evidence present, but no filtered turn lists.
# ============================================================

def print_original_speaks_only_r1_prompt(
    *,
    experiment_title,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert 0 <= case_index < len(ordered_cases)

    case = ordered_cases[case_index]

    baseline_template = (
        load_speaks_only_baseline_prompt_template()
    )

    baseline_payload = (
        build_speaks_only_full_model_input(case)
    )

    baseline_prompt = render_manual_prompt_template(
        baseline_template,
        baseline_payload,
    )

    assert "Filtered turns:" not in baseline_prompt
    assert (
        "local_temporal_features"
        in baseline_payload
    )
    assert set(
        baseline_payload[
            "participant_A"
        ]
    ) == {"speaks"}
    assert set(
        baseline_payload[
            "participant_B"
        ]
    ) == {"speaks"}

    export_dir = (
        OUT_DIR
        / "manual_local_ablation_baseline_prompts"
    )
    export_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        experiment_title.lower(),
    ).strip("_")

    export_path = (
        export_dir
        / f"{safe_name}_original_speaks_only_r1_prompt.txt"
    )

    export_path.write_text(
        baseline_prompt,
        encoding="utf-8",
    )

    print("=" * 100)
    print(
        "ORIGINAL STRUCTURED R1 PROMPT — SPEAKS ONLY"
    )
    print(
        "Target experiment:",
        experiment_title,
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Filtered turns are absent from both the payload and prompt."
    )
    print(
        "This is the unablated local-temporal baseline prompt that "
        "must be edited manually for the target experiment."
    )

    print("\nEXACT BASELINE MODEL-FACING PAYLOAD")
    print(
        json.dumps(
            baseline_payload,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\n" + "=" * 100)
    print("EXACT ORIGINAL RENDERED PROMPT")
    print("=" * 100)
    print(baseline_prompt)

    print("\n" + "=" * 100)
    print("BASELINE HASHES")
    print("=" * 100)
    print(
        "Baseline template SHA256:",
        sha256_text(
            baseline_template
        ),
    )
    print(
        "Rendered baseline prompt SHA256:",
        sha256_text(
            baseline_prompt
        ),
    )
    print(
        "Baseline payload SHA256:",
        sha256_text(
            canonical_json(
                baseline_payload
            )
        ),
    )
    print(
        "Saved rendered prompt:",
        export_path,
    )

    return {
        "case_id": str(
            case[
                "case_id"
            ]
        ),
        "prompt_template": baseline_template,
        "prompt": baseline_prompt,
        "payload": baseline_payload,
        "export_path": export_path,
    }


# ============================================================
# MANUAL PROMPT VALIDATION
#
# This validates the final manually written prompt. It does not edit it.
# ============================================================

def validate_manual_local_prompt_template(
    prompt_template,
    *,
    mode,
):
    assert mode in LOCAL_ABLATION_MODES

    if prompt_template is None:
        raise RuntimeError(
            "The manual prompt template is still None. First run the "
            "ORIGINAL PROMPT cell, send its output for manual editing, "
            "then paste the final complete template into the manual-prompt "
            "cell."
        )

    assert isinstance(prompt_template, str)

    template = prompt_template.strip()

    if not template:
        raise RuntimeError(
            "The manual prompt template is empty."
        )

    placeholder_text = template.upper()
    if (
        "PASTE THE MANUALLY"
        in placeholder_text
        or "PASTE MANUAL"
        in placeholder_text
        or "TODO_MANUAL_PROMPT"
        in placeholder_text
    ):
        raise RuntimeError(
            "The manual prompt placeholder has not been replaced."
        )

    required_placeholders = [
        "{duration_seconds}",
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for placeholder in required_placeholders:
        assert placeholder in template, (
            "Required prompt placeholder missing: "
            f"{placeholder}"
        )

    forbidden_turn_fragments = [
        "Filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "Use the actual filtered turn lists",
        "Use the speaks fields together with the actual filtered turn lists",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
    ]

    for fragment in forbidden_turn_fragments:
        assert fragment not in template, (
            "Filtered-turn instruction or placeholder exists in the "
            f"manual prompt: {fragment}"
        )

    required_general_markers = [
        "PARTICIPATION VALIDITY",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "STRUCTURED REASONING OUTPUT",
        '"participation_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"semantic_assessment"',
        '"decisive_dimension"',
        '"label"',
    ]

    for marker in required_general_markers:
        assert marker in template, (
            "Required Structured R1 marker missing from the manual "
            f"prompt: {marker}"
        )

    if mode == "NO_LOCAL_BRANCH":
        assert "{local_temporal_features}" not in template
        assert "LOCAL TEMPORAL FEATURES" not in template
        assert "FROZEN NORMAL LOCAL-TIMING REFERENCE" not in template
        assert "FILTERED OVERLAP" not in template
        assert "local_temporal_assessment" not in template

    else:
        assert "{local_temporal_features}" in template
        assert "LOCAL TEMPORAL FEATURES" in template
        assert "local_temporal_assessment" in template

    # Render the first case and audit the actual evidence-specific text.
    example_payload = build_manual_local_ablation_model_input(
        consolidation_cases[0],
        mode,
    )

    example_prompt = render_manual_prompt_template(
        template,
        example_payload,
    )

    example_prompt_lower = example_prompt.lower()

    if mode == "NO_LOCAL_BRANCH":
        forbidden_local_fragments = [
            "clean_overlap_seconds",
            "clean_overlap_percent",
            "signed_strict_offsets_seconds",
            "num_signed_strict_offsets",
            "offset_mean_seconds",
            "offset_median_seconds",
            "offset_max_seconds",
            "offset_p75_seconds",
            "offset_p90_seconds",
            "num_offsets_above_1_5_seconds",
            "percent_offsets_above_1_5_seconds",
        ]

        for fragment in forbidden_local_fragments:
            assert fragment not in example_prompt

    elif mode == "NO_OVERLAP":
        assert set(
            example_payload[
                "local_temporal_features"
            ]
        ) == set(
            LOCAL_OFFSET_DISTRIBUTION_FIELDS
        )

        assert "clean_overlap_seconds" not in example_prompt
        assert "clean_overlap_percent" not in example_prompt
        assert "overlap" not in example_prompt_lower, (
            "The manually written L1 prompt still contains overlap "
            "instructions or reference text."
        )

    elif mode == "NO_OFFSET_DISTRIBUTION":
        assert set(
            example_payload[
                "local_temporal_features"
            ]
        ) == set(
            LOCAL_OVERLAP_FIELDS
        )

        for field in LOCAL_OFFSET_DISTRIBUTION_FIELDS:
            assert field not in example_prompt

        forbidden_offset_phrases = [
            "signed offset",
            "signed-offset",
            "b_start minus a_end",
            "offset distribution",
            "offsets above 1.5",
            "offset is around",
            "offsets are above",
        ]

        for phrase in forbidden_offset_phrases:
            assert phrase not in example_prompt_lower, (
                "The manually written L2 prompt still contains an "
                f"offset-dependent instruction: {phrase}"
            )

    return {
        "template": template,
        "example_prompt": example_prompt,
        "example_payload": example_payload,
    }


# ============================================================
# BUILD ONE MANUAL LOCAL-ABLATION PROMPT
# ============================================================

def build_manual_local_ablation_prompt(
    case,
    config,
):
    payload = build_manual_local_ablation_model_input(
        case,
        config[
            "local_ablation_mode"
        ],
    )

    prompt = render_manual_prompt_template(
        config[
            "reasoning_prompt_template"
        ],
        payload,
    )

    return prompt, payload


# ============================================================
# PREPARE ONE MANUAL LOCAL-ABLATION EXPERIMENT
#
# The function saves exactly the supplied template. It does not perform
# replacements, section deletion, or any other prompt transformation.
# ============================================================

def prepare_manual_local_ablation_experiment(
    *,
    experiment_version,
    experiment_title,
    ablation_id,
    local_ablation_mode,
    manual_prompt_template,
):
    assert local_ablation_mode in LOCAL_ABLATION_MODES

    validated = validate_manual_local_prompt_template(
        manual_prompt_template,
        mode=local_ablation_mode,
    )

    ablation_prompt_template = validated[
        "template"
    ]

    baseline_prompt_template = (
        load_speaks_only_baseline_prompt_template()
    )

    if local_ablation_mode == "NO_LOCAL_BRANCH":
        schema_keys = list(
            NO_LOCAL_SCHEMA_KEYS
        )
        allowed_values = copy.deepcopy(
            NO_LOCAL_ALLOWED_VALUES
        )
        output_block = (
            NO_LOCAL_REASONING_OUTPUT_BLOCK
        )
    else:
        schema_keys = list(
            FULL_LOCAL_SCHEMA_KEYS
        )
        allowed_values = copy.deepcopy(
            FULL_LOCAL_ALLOWED_VALUES
        )
        output_block = (
            STRUCTURED_REASONING_OUTPUT_BLOCK
        )

    experiment_dir = (
        OUT_DIR
        / experiment_version
    )
    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),
        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),
        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),
        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),
        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),
        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),
        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),
        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),
        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),
        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),
        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),
        "baseline_prompt_copy": (
            experiment_dir
            / "original_speaks_only_structured_r1_prompt_template.txt"
        ),
        "prompt_diff": (
            experiment_dir
            / "manual_prompt_diff_vs_speaks_only_baseline.txt"
        ),
        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }

    paths[
        "prompt_template"
    ].write_text(
        ablation_prompt_template,
        encoding="utf-8",
    )

    paths[
        "baseline_prompt_copy"
    ].write_text(
        baseline_prompt_template,
        encoding="utf-8",
    )

    prompt_diff_lines = difflib.unified_diff(
        baseline_prompt_template.splitlines(),
        ablation_prompt_template.splitlines(),
        fromfile=(
            "original_speaks_only_structured_r1_prompt_template.txt"
        ),
        tofile=(
            "manually_defined_ablation_prompt_template.txt"
        ),
        lineterm="",
    )

    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(prompt_diff_lines),
        encoding="utf-8",
    )

    baseline_prompt_sha256 = sha256_text(
        baseline_prompt_template
    )
    reasoning_prompt_sha256 = sha256_text(
        ablation_prompt_template
    )
    reasoning_schema_sha256 = sha256_text(
        output_block
    )
    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )

    manifest = {
        "experiment_version": experiment_version,
        "experiment_title": experiment_title,
        "ablation_id": ablation_id,
        "local_ablation_mode": (
            local_ablation_mode
        ),
        "source_experiment": (
            SPEAKS_ONLY_BASELINE_EXPERIMENT_NAME
        ),
        "source_prompt_sha256": (
            baseline_prompt_sha256
        ),
        "full_r1_prompt_sha256": (
            baseline_prompt_sha256
        ),
        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),
        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),
        "normal_reference_sha256": (
            normal_reference_sha256
        ),
        "semantic_input": (
            "coarse_and_focused"
        ),
        "focused_summaries_used": True,
        "temporal_profiles_used": [
            "NORMAL"
        ],
        "assessment_policy": (
            "Structured R1 policy using the validated speaks-only "
            "participation representation. The local-ablation prompt "
            "was defined manually and was not programmatically rewritten."
        ),
        "schema_keys": schema_keys,
        "allowed_values": {
            key: sorted(list(values))
            for key, values
            in allowed_values.items()
        },
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "model_id": MODEL_ID,
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
        "participant_model_input": (
            "speaks_only"
        ),
        "filtered_turns_excluded_from_all_model_facing_inputs": True,
        "prompt_definition_method": (
            "manual_complete_template"
        ),
        "automatic_prompt_rewriting_used": False,
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": experiment_dir,
        "baseline_prompt_template": (
            baseline_prompt_template
        ),
        "reasoning_prompt_template": (
            ablation_prompt_template
        ),
        "paths": paths,
    }

    example_prompt, example_payload = (
        build_manual_local_ablation_prompt(
            consolidation_cases[0],
            config,
        )
    )

    assert set(
        example_payload[
            "participant_A"
        ]
    ) == {"speaks"}
    assert set(
        example_payload[
            "participant_B"
        ]
    ) == {"speaks"}
    assert "Filtered turns:" not in example_prompt

    print("=" * 88)
    print(
        f"{experiment_title} — MANUAL CONFIGURATION READY"
    )
    print("=" * 88)
    print(
        "Ablation ID:",
        ablation_id,
    )
    print(
        "Local ablation mode:",
        local_ablation_mode,
    )
    print(
        "Participant evidence:",
        "speaks only",
    )
    print(
        "Filtered turns passed to model:",
        False,
    )
    print(
        "Automatic prompt rewriting used:",
        False,
    )
    print(
        "Schema keys:",
        schema_keys,
    )
    print(
        "Baseline prompt SHA256:",
        baseline_prompt_sha256,
    )
    print(
        "Manual prompt SHA256:",
        reasoning_prompt_sha256,
    )
    print(
        "Prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    return config


# ============================================================
# REQUIRED MANUAL PROMPT INSPECTION
# ============================================================

def inspect_manual_local_ablation_prompt(
    config,
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert 0 <= case_index < len(ordered_cases)

    case = ordered_cases[case_index]

    prompt, payload = (
        build_manual_local_ablation_prompt(
            case,
            config,
        )
    )

    print("=" * 100)
    print(
        "MANUAL PROMPT INSPECTION —",
        config[
            "experiment_title"
        ],
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Gold label is intentionally NOT printed inside the model prompt."
    )
    print(
        "Participant evidence supplied: speaks only."
    )
    print(
        "Filtered turns supplied: False"
    )
    print(
        "Prompt definition method: manual complete template"
    )

    print("\nEXACT ABLATED MODEL INPUT PAYLOAD")
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\n" + "=" * 100)
    print("EXACT MANUALLY DEFINED RENDERED MODEL PROMPT")
    print("=" * 100)
    print(prompt)

    print("\n" + "=" * 100)
    print("PROMPT INSPECTION HASHES")
    print("=" * 100)
    print(
        "Rendered prompt SHA256:",
        sha256_text(prompt),
    )
    print(
        "Ablated payload SHA256:",
        sha256_text(
            canonical_json(payload)
        ),
    )
    print(
        "Manual template SHA256:",
        config[
            "reasoning_prompt_sha256"
        ],
    )
    print(
        "Template diff file:",
        config[
            "paths"
        ][
            "prompt_diff"
        ],
    )

    INSPECTED_MANUAL_LOCAL_ABLATIONS.add(
        config[
            "experiment_version"
        ]
    )

    return {
        "case_id": str(
            case[
                "case_id"
            ]
        ),
        "prompt": prompt,
        "payload": payload,
    }


# ============================================================
# CONFIG-AWARE STRUCTURED OUTPUT PARSER
# ============================================================

def parse_manual_local_ablation_prediction(
    raw_output,
    config,
):
    parsed = extract_first_json_object(
        raw_output
    )

    schema_keys = config[
        "schema_keys"
    ]

    allowed_values = {
        key: set(values)
        for key, values
        in config[
            "allowed_values"
        ].items()
    }

    normalized = {
        key: None
        for key in schema_keys
    }

    schema_errors = []

    if isinstance(parsed, dict):
        for key in schema_keys:
            if key in parsed:
                normalized[key] = str(
                    parsed[key]
                ).strip().upper()

        for key in schema_keys:
            if normalized[key] not in allowed_values[key]:
                schema_errors.append(
                    f"{key}: {normalized[key]}"
                )

        exact_keys = (
            set(parsed.keys())
            == set(schema_keys)
        )

        schema_exact = (
            exact_keys
            and not schema_errors
        )

        label = normalized[
            "label"
        ]

        if label in LABELS:
            result = {
                "prediction": label,
                "parse_mode": (
                    "structured_json"
                ),
                "schema_exact": bool(
                    schema_exact
                ),
                "parsed_output": parsed,
                "schema_errors": (
                    schema_errors
                ),
            }

            for field in (
                STANDARD_REASONING_FIELDS_LOCAL_ABLATION
            ):
                result[field] = normalized.get(field)

            return result

    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )

    if plain_output in LABELS:
        return {
            "prediction": plain_output,
            "parse_mode": (
                "exact_plaintext_fallback"
            ),
            "schema_exact": False,
            "parsed_output": None,
            "schema_errors": [
                "Structured reasoning fields missing."
            ],
            **{
                field: None
                for field in (
                    STANDARD_REASONING_FIELDS_LOCAL_ABLATION
                )
            },
        }

    return {
        "prediction": None,
        "parse_mode": "invalid",
        "schema_exact": False,
        "parsed_output": parsed,
        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),
        **{
            field: normalized.get(field)
            for field in (
                STANDARD_REASONING_FIELDS_LOCAL_ABLATION
            )
        },
    }


# ============================================================
# CHECKPOINT CACHE
# ============================================================

def create_manual_local_ablation_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),
        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),
        "ablation_id": (
            config[
                "ablation_id"
            ]
        ),
        "local_ablation_mode": (
            config[
                "local_ablation_mode"
            ]
        ),
        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),
        "model_id": MODEL_ID,
        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),
        "full_r1_prompt_sha256": (
            config[
                "full_r1_prompt_sha256"
            ]
        ),
        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),
        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),
        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),
        "semantic_input": (
            "coarse_and_focused"
        ),
        "focused_summaries_used": True,
        "temporal_profiles_used": [
            "NORMAL"
        ],
        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),
        "schema_keys": (
            config[
                "schema_keys"
            ]
        ),
        "participant_model_input": (
            "speaks_only"
        ),
        "filtered_turns_excluded": True,
        "prompt_definition_method": (
            "manual_complete_template"
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "created_at_utc": (
            reasoning_utc_now()
        ),
        "updated_at_utc": (
            reasoning_utc_now()
        ),
        "records": {},
    }


# ============================================================
# INFERENCE — SAME QWEN CALL / ORDER / DECODING / CHECKPOINTING
# ============================================================

def run_manual_local_ablation_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config[
            "experiment_version"
        ]
        in INSPECTED_MANUAL_LOCAL_ABLATIONS
    ), (
        "Manual prompt inspection has not been completed in this "
        "runtime. Run the inspection cell immediately above before "
        "starting inference."
    )

    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )

    expected_cache_header = (
        create_manual_local_ablation_cache(
            config
        )
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "local_ablation_mode",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "full_r1_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "schema_keys",
            "participant_model_input",
            "filtered_turns_excluded",
            "prompt_definition_method",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[key]
                == expected_cache_header[key]
            ), (
                f"Cache mismatch for {key}. Delete the old cache "
                "only if you intentionally changed the manual prompt."
            )

        print(
            "Resuming cache:",
            prediction_cache_path,
        )
        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )

    else:
        prediction_cache = expected_cache_header
        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )
        print(
            "Created cache:",
            prediction_cache_path,
        )

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(ordered_cases) == 400

    for case in tqdm(
        ordered_cases,
        desc=config[
            "experiment_version"
        ],
    ):
        case_id = str(
            case[
                "case_id"
            ]
        )

        prompt, model_input_payload = (
            build_manual_local_ablation_prompt(
                case,
                config,
            )
        )

        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )

        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):
            assert required_key in current_payload_keys

        speaks_only_full_payload = (
            build_speaks_only_full_model_input(
                case
            )
        )

        # Exact invariance checks for all retained branches.
        assert (
            model_input_payload[
                "participant_A"
            ]
            == speaks_only_full_payload[
                "participant_A"
            ]
        )
        assert (
            model_input_payload[
                "participant_B"
            ]
            == speaks_only_full_payload[
                "participant_B"
            ]
        )
        assert (
            model_input_payload[
                "global_shift_features"
            ]
            == speaks_only_full_payload[
                "global_shift_features"
            ]
        )
        assert (
            model_input_payload[
                "semantic_summaries"
            ]
            == speaks_only_full_payload[
                "semantic_summaries"
            ]
        )

        assert set(
            model_input_payload[
                "participant_A"
            ]
        ) == {"speaks"}
        assert set(
            model_input_payload[
                "participant_B"
            ]
        ) == {"speaks"}
        assert "Filtered turns:" not in prompt

        mode = config[
            "local_ablation_mode"
        ]

        if mode == "NO_LOCAL_BRANCH":
            assert (
                "local_temporal_features"
                not in model_input_payload
            )
        elif mode == "NO_OVERLAP":
            assert set(
                model_input_payload[
                    "local_temporal_features"
                ]
            ) == set(
                LOCAL_OFFSET_DISTRIBUTION_FIELDS
            )
        elif mode == "NO_OFFSET_DISTRIBUTION":
            assert set(
                model_input_payload[
                    "local_temporal_features"
                ]
            ) == set(
                LOCAL_OVERLAP_FIELDS
            )
        else:
            raise ValueError(
                f"Unknown local ablation mode: {mode}"
            )

        prompt_sha256 = sha256_text(prompt)
        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )

        existing_record = prediction_cache[
            "records"
        ].get(case_id)

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )
            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )
            continue

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print("CASE:", case_id)
            print("=" * 100)
            print(prompt)

        started = time.perf_counter()

        try:
            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_manual_local_ablation_prediction(
                    raw_output,
                    config,
                )
            )
            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None
            parsed_result = {
                "prediction": None,
                "parse_mode": (
                    "generation_error"
                ),
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                **{
                    field: None
                    for field in (
                        STANDARD_REASONING_FIELDS_LOCAL_ABLATION
                    )
                },
            }
            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),
            "case_family": get_case_family(case),
            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),
            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),
            "ablation_id": (
                config[
                    "ablation_id"
                ]
            ),
            "local_ablation_mode": mode,
            "participant_model_input": (
                "speaks_only"
            ),
            "filtered_turns_excluded": True,
            "prompt_definition_method": (
                "manual_complete_template"
            ),
            "prompt_sha256": prompt_sha256,
            "input_payload_sha256": (
                input_payload_sha256
            ),
            "semantic_input": (
                "coarse_and_focused"
            ),
            "focused_summaries_used": True,
            "input_token_count": (
                input_token_count
            ),
            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),
            "raw_output": raw_output,
            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),
            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),
            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),
            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),
            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),
            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),
            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),
            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),
            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),
            "generation_error": generation_error,
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)
    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )
    print(
        "Prediction cache:",
        prediction_cache_path,
    )

    return prediction_cache


# Required workflow

For each experiment, execute the cells in this order:

1. **Print original prompt** — this produces the full speaks-only Structured R1 baseline prompt for one real case.
2. Send that printed prompt for manual adaptation.
3. Replace the `None` value in the **manual prompt template** cell with the complete adapted template.
4. Run configuration.
5. Run prompt inspection.
6. Only then run the 400-case inference and evaluation.

A manual prompt template must retain Python placeholders such as `{duration_seconds}`, `{participant_A_speaks}`, `{global_shift_features}` and `{semantic_summaries}`. Literal JSON braces in the output schema must be doubled as `{{` and `}}`.


# A2 — No Local Temporal Branch

**Model-facing evidence:** `speaks`, global temporal features and semantic summaries. No filtered turns and no local temporal feature object.

**Output:** remove `local_temporal_assessment`; retain participation, global temporal, combined temporal, semantic, decisive dimension and final label.


In [ ]:
# STEP A2.1 — PRINT THE ORIGINAL SPEAKS-ONLY STRUCTURED R1 PROMPT
# Run only this cell first and send the printed prompt for manual adaptation.

A2_ORIGINAL_PROMPT = print_original_speaks_only_r1_prompt(
    experiment_title=(
        "A2 — No Local Temporal Branch"
    ),
    case_index=0,
)


ORIGINAL STRUCTURED R1 PROMPT — SPEAKS ONLY
Target experiment: A2 — No Local Temporal Branch
Inspection case ID: consolidation_lag_2sec_000
Filtered turns are absent from both the payload and prompt.
This is the unablated local-temporal baseline prompt that must be edited manually for the target experiment.

EXACT BASELINE MODEL-FACING PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "clean_overlap_seconds": 0.0,
    "clean_overlap_percent": 0.0,
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offsets_above_1_5_seconds": 2,
    "percent_offsets_above_1_5_seconds": 100.0
  },
  "global_shift_features": {
    "best_B_correction_shift_seconds

In [ ]:
# STEP A2.2 — MANUAL PROMPT TEMPLATE
# Leave this as None until the complete A2 prompt has been manually defined.
# Then replace None with:
#
# A2_MANUAL_PROMPT_TEMPLATE = r"""
# <complete manually adapted prompt template>
# """.strip()

A2_MANUAL_PROMPT_TEMPLATE = r"""
You are evaluating whether two participant records are jointly
consistent with one coherent, naturally synchronized, spoken
120-second dyadic interaction.

You are given information extracted independently for Participant A
and Participant B from the same 120-second analysis timeline.

Your task is binary classification only:

- NORMAL
- ANOMALOUS

Do not predict, infer, or name a specific anomaly type.

============================================================
AVAILABLE EVIDENCE
============================================================

You receive:

1. Whether each participant speaks during the 120-second interval.
2. Global temporal alignment-shift features.
3. Coarse semantic summaries for two synchronized 60-second segments.
4. Focused semantic summaries for the same two segments.
5. Frozen global temporal reference statistics calculated only from
   separate NORMAL dyadic conversations.

Use only the supplied evidence.

Do not infer anything from identifiers, filenames, paths, dataset order,
sample position, participant identity, or hidden labels.

None of those fields are provided.

============================================================
OPERATIONAL DEFINITION OF NORMAL
============================================================

A NORMAL case must be compatible with one coherent, naturally
synchronized, two-person spoken interaction.

NORMAL requires three independent properties:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Evaluate these three properties separately before making the final
binary decision.

A case should be classified as NORMAL only when all three properties
are sufficiently supported by the available evidence.

A strong and reliable failure of any one property means that the
complete interaction does not satisfy the operational definition of
NORMAL.

Do not use a simple majority vote between the three properties.

Evidence that two properties appear normal must not override a strong
and reliable failure of the third property.

In particular:

- Both participants speaking does not by itself prove NORMAL.
- Semantic compatibility does not by itself prove NORMAL.
- Temporal compatibility does not compensate for strong semantic
  incompatibility.
- Strong semantic compatibility must not cancel reliable evidence
  that the participant timelines are not normally coordinated.

============================================================
PARTICIPATION VALIDITY
============================================================

The field "speaks" is derived directly from the final filtered VAD turns:

- speaks = true means that the participant has at least one retained turn.
- speaks = false means that the participant has no retained turns during
  the entire 120-second interval.

Use the speaks fields.

Do not infer speaking activity from the semantic summaries.

For a normal spoken dyadic interaction, both participants are expected
to contribute speech during the complete interval.

Natural asymmetry is allowed:

- one participant may speak substantially more than the other,
- participants may have long listening periods,
- turn numbers and speaking durations do not need to be balanced.

However, the complete absence of retained speech from one participant
is not compatible with a normal two-person spoken interaction.

============================================================
GLOBAL ALIGNMENT-SHIFT FEATURES
============================================================

The global alignment search evaluates hypothetical temporal corrections
to Participant B.

The global features have the following meanings:

- best_B_correction_shift_seconds is the hypothetical correction that
  produced the strongest bilateral turn-boundary alignment.

- A correction near zero means that little global temporal correction
  was preferred.

- A negative correction means that Participant B would align better if
  Participant B's timeline were moved earlier.

- estimated_B_lateness_seconds is the non-negative lateness implied by
  the best correction.

- alignment_score_gain_vs_zero measures how much the best correction
  improves alignment compared with applying no correction.

- best_num_bilateral_events indicates how many bilateral events support
  the selected correction.

- best_event_coverage indicates how much of the available interaction
  supports that estimate.

Do not use the correction value alone.

A large correction based on very few events or very low event coverage
is weak evidence.

A global estimate becomes more reliable when it is jointly supported by:

- a meaningful correction,
- meaningful estimated lateness,
- meaningful alignment improvement over zero shift,
- several bilateral events,
- and sufficiently broad event coverage.

============================================================
FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE
============================================================

The following statistics were calculated only from the same frozen
NORMAL reference set:

Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.
- Median alignment score gain versus zero shift is around 0.009.
- Mean number of bilateral alignment events is around 11.94.
- Mean bilateral event coverage is around 59.52%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.

These statistics operationally describe the expected global alignment
behavior for NORMAL interactions in this experiment.

They are not independent hard thresholds.

Natural variation is allowed, and one unusual global value does not
automatically exclude NORMAL.

For NORMAL temporal coordination, the global evidence is generally
expected to show:

- a preferred correction near the frozen NORMAL pattern,
- limited estimated lateness,
- limited improvement over applying no correction,
- or insufficient reliable evidence that a substantial correction
  is necessary.

The global temporal requirement for NORMAL is not satisfied when a
substantial correction is reliably supported by the evidence as a whole.

Evaluate together:

- correction direction and magnitude,
- estimated lateness,
- alignment-score gain over zero shift,
- number of bilateral events,
- and event coverage.

A substantial correction with negligible gain, very few events, or
very low coverage is weak evidence.

A substantial correction with meaningful gain, sufficient bilateral
events, and broad coverage is strong evidence that the interaction does
not follow the frozen NORMAL temporal pattern.

============================================================
TEMPORAL COORDINATION DECISION
============================================================

Global temporal evidence must be used to evaluate temporal coordination.

Temporal coordination is a necessary and independent property of a
NORMAL interaction.

A case can fail the temporal requirement for NORMAL even when:

- both participants speak,
- their semantic content is compatible,
- they discuss the same topic,
- they refer to the same people or events.

Semantic compatibility establishes content compatibility.
It does not establish temporal synchronization.

Treat the temporal dimension as compatible with NORMAL when:

- the preferred global correction is near the frozen NORMAL pattern,
- estimated lateness and alignment improvement are limited,
- or a larger correction is weakly supported by gain, events, or coverage.

Treat the temporal dimension as not compatible with NORMAL when:

- a substantial correction is reliably supported by meaningful
  alignment improvement,
- several bilateral events,
- and sufficiently broad event coverage.

Do not require every global temporal feature to depart from NORMAL.

Do not classify a case as ANOMALOUS because of one isolated unusual
global value.

When reliable global temporal evidence indicates that the interaction
substantially departs from the frozen NORMAL pattern, the complete case
must be classified as ANOMALOUS, even when the semantic evidence is
fully compatible.

Do not infer or report the cause or magnitude of the temporal failure.

============================================================
SEMANTIC EVIDENCE
============================================================

The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and segment, you receive:

Coarse semantic information:

- speech_content_summary
- apparent_topic

Focused semantic information:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The semantic summaries were independently generated and may be broad,
imperfect, repetitive, or uncertain.

Evaluate semantic compatibility rather than exact wording.

A NORMAL semantic relationship may include:

- a shared concrete subject,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant supplying context for the other,
- one participant elaborating on or reacting to the other,
- compatible people, events, places, experiences, or arguments,
- different aspects of a shared subject,
- a coherent topic transition between Segment 0 and Segment 1.

Exact word matching and identical topic labels are not required.

A conversation may naturally change topic during the 120 seconds.

Broad labels such as:

- personal experiences,
- preferences,
- daily life,
- opinions,
- general discussion,
- lifestyle,
- personal well-being

are not sufficient evidence of semantic compatibility by themselves.

Concrete content must provide a plausible shared conversational context.

If one summary is vague, generic, unclear, or low-confidence, treat it
as limited evidence rather than automatically supporting either label.

Do not interpret the fact that both participants discuss generic
personal topics as sufficient evidence that they belong to one coherent
conversation.

Evaluate both synchronized segments and the complete 120-second semantic
relationship.

Do not use a simple vote between Segment 0 and Segment 1.

============================================================
FINAL COMBINED DECISION
============================================================

Internally evaluate:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Then make one binary decision.

Classify as NORMAL only when the complete evidence is sufficiently
compatible with all three required properties of a normal dyadic
interaction.

Classify as ANOMALOUS when at least one required property shows a strong,
reliable, and well-supported failure.

Do not require multiple properties to fail.

Do not allow strong evidence from one property to erase a reliable
failure in another property.

In particular:

- Semantic compatibility must not cancel reliable temporal failure.
- Temporal compatibility must not cancel strong semantic incompatibility.
- Speech from both participants must not cancel semantic or temporal failure.

Do not classify as ANOMALOUS because of one isolated noisy measurement.

Do not classify as NORMAL merely because one evidence source appears
plausible.

Use the reliability and consistency of the evidence, not a simple count
of supportive features.

Return one final label even when some evidence is limited.

============================================================
CURRENT CASE
============================================================

Analysis duration:
{duration_seconds} seconds

PARTICIPANT A

Speaks:
{participant_A_speaks}

PARTICIPANT B

Speaks:
{participant_B_speaks}

GLOBAL ALIGNMENT-SHIFT FEATURES

{global_shift_features}

SEMANTIC SUMMARIES

{semantic_summaries}

============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The global temporal assessment must reflect the reliability rules
already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


In [ ]:
# STEP A2.3 — PREPARE THE EXACT MANUALLY DEFINED EXPERIMENT

A2_NO_LOCAL_CONFIG = prepare_manual_local_ablation_experiment(
    experiment_version=(
        "manual_ablation_a2_no_local_temporal_branch_speaks_only"
    ),
    experiment_title=(
        "A2 — No Local Temporal Branch — Manual Prompt — Speaks Only"
    ),
    ablation_id=(
        "A2_NO_LOCAL_TEMPORAL_BRANCH_MANUAL_SPEAKS_ONLY"
    ),
    local_ablation_mode=(
        "NO_LOCAL_BRANCH"
    ),
    manual_prompt_template=(
        A2_MANUAL_PROMPT_TEMPLATE
    ),
)


A2 — No Local Temporal Branch — Manual Prompt — Speaks Only — MANUAL CONFIGURATION READY
Ablation ID: A2_NO_LOCAL_TEMPORAL_BRANCH_MANUAL_SPEAKS_ONLY
Local ablation mode: NO_LOCAL_BRANCH
Participant evidence: speaks only
Filtered turns passed to model: False
Automatic prompt rewriting used: False
Schema keys: ['participation_assessment', 'global_temporal_assessment', 'temporal_assessment', 'semantic_assessment', 'decisive_dimension', 'label']
Baseline prompt SHA256: cb64963d1881e6b5b14de4088dab4fcf97e95dbfa5c02bdc07051be896366b5c
Manual prompt SHA256: 505c41ab3724a8f0347f7db8c9e3b916ff9f1c3f8b2bf8e3b48fa76d6a38fe08
Prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/manual_prompt_diff_vs_speaks_only_baseline.txt


In [ ]:
# STEP A2.4 — REQUIRED PROMPT INSPECTION

A2_NO_LOCAL_INSPECTION = inspect_manual_local_ablation_prompt(
    A2_NO_LOCAL_CONFIG,
    case_index=0,
)


MANUAL PROMPT INSPECTION — A2 — No Local Temporal Branch — Manual Prompt — Speaks Only
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.
Participant evidence supplied: speaks only.
Filtered turns supplied: False
Prompt definition method: manual complete template

EXACT ABLATED MODEL INPUT PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "global_shift_features": {
    "best_B_correction_shift_seconds": 2.7,
    "estimated_B_lateness_seconds": 0.0,
    "alignment_score_gain_vs_zero": 0.069487,
    "best_num_bilateral_events": 5,
    "best_event_coverage_percent": 23.8095
  },
  "semantic_summaries": {
    "participant_A": {
      "segment_0": {
        "coarse_summary": {
          "speech_content_summary": "talking about a difficult situation involving a child and a parent's death",
          "apparent_topic": "personal experiences and 

In [ ]:
# STEP A2.5 — RUN THE 400-CASE EXPERIMENT
# Set print_each_case_prompt=True only if all 400 prompts should be printed.

A2_NO_LOCAL_CACHE = run_manual_local_ablation_experiment(
    A2_NO_LOCAL_CONFIG,
    print_each_case_prompt=False,
)


Resuming cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/predictions_cache.json
Existing records: 400


manual_ablation_a2_no_local_temporal_branch_speaks_only:   0%|          | 0/400 [00:00<?, ?it/s]


A2 — No Local Temporal Branch — Manual Prompt — Speaks Only — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/predictions_cache.json


In [ ]:
# STEP A2.6 — EVALUATE

A2_NO_LOCAL_EVALUATION = evaluate_reasoning_experiment(
    A2_NO_LOCAL_CONFIG
)


A2 — No Local Temporal Branch — Manual Prompt — Speaks Only — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8375
Balanced accuracy: 0.6817
ANOMALOUS precision: 0.8255
ANOMALOUS recall: 0.9933
ANOMALOUS F1: 0.9017
NORMAL recall / specificity: 0.3700
MCC: 0.5304
Matched source-group exact rate: 0.3600
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,37,63
Gold ANOMALOUS,2,298



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.948718,0.370000,0.532374,100.0000
ANOMALOUS,0.825485,0.993333,0.901664,300.0000
accuracy,0.837500,0.837500,0.837500,0.8375
macro avg,0.887101,0.681667,0.717019,400.0000
weighted avg,0.856293,0.837500,0.809342,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,0,50,50,1.00,1.00,1.0
1,lag_3sec,50,50,0,1,49,49,0.98,0.98,1.0
2,normal,100,100,0,37,63,37,0.37,0.37,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,1,99,99,0.99,0.99,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,MISSING,400
3,global_temporal_assessment,ANOMALOUS,358
4,global_temporal_assessment,NORMAL,40
5,global_temporal_assessment,LIMITED,2
6,temporal_assessment,ANOMALOUS,358
7,temporal_assessment,NORMAL,40
8,temporal_assessment,LIMITED,2
9,semantic_assessment,COMPATIBLE,257



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/manual_prompt_diff_vs_speaks_only_baseline.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_a2_no_local_temporal_branch_speaks_only/classificatio

In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD A2 — NO LOCAL TEMPORAL BRANCH
# ============================================================

if "A2_NO_LOCAL_EVALUATION" in globals():

    a2_df = (
        A2_NO_LOCAL_EVALUATION[
            "results_df"
        ].copy()
    )

elif "A2_NO_LOCAL_CONFIG" in globals():

    a2_df = pd.read_csv(
        A2_NO_LOCAL_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the A2_NO_LOCAL configuration "
        "and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in a2_df.columns:

        a2_df[column] = (
            a2_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in a2_df.columns:

        a2_df[column] = (
            a2_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if "valid_prediction" not in a2_df.columns:

    a2_df["valid_prediction"] = (
        a2_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in a2_df.columns:

    a2_df["correct"] = (
        a2_df["valid_prediction"]
        &
        (
            a2_df["gold_label"]
            ==
            a2_df["prediction"]
        )
    )


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
#
# local_temporal_assessment is intentionally absent in A2.
# ============================================================

A2_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def a2_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(unexpected_values)
    )


    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_a2_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[row_field].fillna("MISSING"),
            subset[column_field].fillna("MISSING"),
            margins=True,
        )
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_a2_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset["gold_label"]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset["gold_label"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset["prediction"]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset["prediction"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset["valid_prediction"]
            ).sum()
        ),

        "correct": int(
            subset["correct"].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset["correct"].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    if "case_variant" in subset.columns:

        print(
            "\nCASE VARIANT BREAKDOWN"
        )


        variant_counts = (
            subset["case_variant"]
            .fillna("MISSING")
            .value_counts()
            .rename_axis("case_variant")
            .reset_index(name="count")
        )


        variant_counts["percentage"] = (
            100.0
            *
            variant_counts["count"]
            /
            len(subset)
        ).round(2)


        display(
            variant_counts
        )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            a2_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )
            for field, expected_values
            in A2_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_a2_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a2_crosstab(
        subset,
        "global_temporal_assessment",
        "temporal_assessment",
        (
            "GLOBAL TEMPORAL ASSESSMENT "
            "× TEMPORAL ASSESSMENT"
        ),
    )


    display_a2_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "TEMPORAL ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a2_crosstab(
        subset,
        "semantic_assessment",
        "decisive_dimension",
        (
            "SEMANTIC ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a2_crosstab(
        subset,
        "temporal_assessment",
        "semantic_assessment",
        (
            "TEMPORAL × SEMANTIC ASSESSMENT"
        ),
    )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_a2_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe["case_family"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE OUTCOME SUBSET
# ============================================================

def get_a2_outcome_subset(
    family,
    outcome,
):

    family_mask = get_a2_family_mask(
        dataframe=a2_df,
        family=family,
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )


    return a2_df[
        family_mask
        &
        (
            a2_df["gold_label"]
            ==
            gold_label
        )
        &
        (
            a2_df["prediction"]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "A2 cases loaded:",
    len(a2_df),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    a2_df["case_family"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("case_family")
    .reset_index(name="count")
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        a2_df["case_family"],
        a2_df["prediction"],
        margins=True,
    )
)

A2 cases loaded: 400

CASE FAMILY COUNTS


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,99,1,100
normal,63,37,100
silent_partner,100,0,100
wrong_partner,99,1,100
All,361,39,400


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

a2_lag_correct = get_a2_outcome_subset(
    family="lag",
    outcome="correct",
)

inspect_a2_subset(
    a2_lag_correct,
    (
        "A2 — LAG CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

a2_lag_missed = get_a2_outcome_subset(
    family="lag",
    outcome="missed",
)

inspect_a2_subset(
    a2_lag_missed,
    (
        "A2 — LAG CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

a2_wrong_partner_correct = get_a2_outcome_subset(
    family="wrong_partner",
    outcome="correct",
)

inspect_a2_subset(
    a2_wrong_partner_correct,
    (
        "A2 — WRONG-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

a2_wrong_partner_missed = get_a2_outcome_subset(
    family="wrong_partner",
    outcome="missed",
)

inspect_a2_subset(
    a2_wrong_partner_missed,
    (
        "A2 — WRONG-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

a2_normal_correct = get_a2_outcome_subset(
    family="normal",
    outcome="correct",
)

inspect_a2_subset(
    a2_normal_correct,
    (
        "A2 — NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

a2_normal_missed = get_a2_outcome_subset(
    family="normal",
    outcome="missed",
)

inspect_a2_subset(
    a2_normal_missed,
    (
        "A2 — NORMAL CASES INCORRECTLY "
        "PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

a2_silent_partner_correct = get_a2_outcome_subset(
    family="silent_partner",
    outcome="correct",
)

inspect_a2_subset(
    a2_silent_partner_correct,
    (
        "A2 — SILENT-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

a2_silent_partner_missed = get_a2_outcome_subset(
    family="silent_partner",
    outcome="missed",
)

inspect_a2_subset(
    a2_silent_partner_missed,
    (
        "A2 — SILENT-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


A2 — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,99,0,99,0,99,0,99,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_2sec,50,50.51
1,lag_3sec,49,49.49



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,All
semantic_assessment,,
COMPATIBLE,24,24
LIMITED,76,76
All,100,100



TEMPORAL × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,LIMITED,All
temporal_assessment,,,
ANOMALOUS,24,76,100
All,24,76,100



A2 — SILENT-PARTNER CASES NOT DETECTED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


# L1 — No Overlap Evidence

**Model-facing local evidence:** the complete offset-distribution group only.

Removed:

- `clean_overlap_seconds`
- `clean_overlap_percent`

`local_temporal_assessment` remains in the structured output. Participant evidence is still `speaks` only.


In [ ]:
# STEP L1.1 — PRINT THE ORIGINAL SPEAKS-ONLY STRUCTURED R1 PROMPT
# Run only this cell first and send the printed prompt for manual adaptation.

L1_ORIGINAL_PROMPT = print_original_speaks_only_r1_prompt(
    experiment_title=(
        "L1 — No Overlap Evidence"
    ),
    case_index=0,
)


ORIGINAL STRUCTURED R1 PROMPT — SPEAKS ONLY
Target experiment: L1 — No Overlap Evidence
Inspection case ID: consolidation_lag_2sec_000
Filtered turns are absent from both the payload and prompt.
This is the unablated local-temporal baseline prompt that must be edited manually for the target experiment.

EXACT BASELINE MODEL-FACING PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "clean_overlap_seconds": 0.0,
    "clean_overlap_percent": 0.0,
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offsets_above_1_5_seconds": 2,
    "percent_offsets_above_1_5_seconds": 100.0
  },
  "global_shift_features": {
    "best_B_correction_shift_seconds": 2.

In [ ]:
# STEP L1.2 — MANUAL PROMPT TEMPLATE
# Leave this as None until the complete L1 prompt has been manually defined.

L1_MANUAL_PROMPT_TEMPLATE = r"""
You are evaluating whether two participant records are jointly
consistent with one coherent, naturally synchronized, spoken
120-second dyadic interaction.

You are given information extracted independently for Participant A
and Participant B from the same 120-second analysis timeline.

Your task is binary classification only:

- NORMAL
- ANOMALOUS

Do not predict, infer, or name a specific anomaly type.

============================================================
AVAILABLE EVIDENCE
============================================================

You receive:

1. Whether each participant speaks during the 120-second interval.
2. Local turn-handoff offset-distribution features.
3. Global temporal alignment-shift features.
4. Coarse semantic summaries for two synchronized 60-second segments.
5. Focused semantic summaries for the same two segments.
6. Frozen temporal reference statistics calculated only from
   separate NORMAL dyadic conversations.

Use only the supplied evidence.

Do not infer anything from identifiers, filenames, paths, dataset order,
sample position, participant identity, or hidden labels.

None of those fields are provided.

============================================================
OPERATIONAL DEFINITION OF NORMAL
============================================================

A NORMAL case must be compatible with one coherent, naturally
synchronized, two-person spoken interaction.

NORMAL requires three independent properties:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Evaluate these three properties separately before making the final
binary decision.

A case should be classified as NORMAL only when all three properties
are sufficiently supported by the available evidence.

A strong and reliable failure of any one property means that the
complete interaction does not satisfy the operational definition of
NORMAL.

Do not use a simple majority vote between the three properties.

Evidence that two properties appear normal must not override a strong
and reliable failure of the third property.

In particular:

- Both participants speaking does not by itself prove NORMAL.
- Semantic compatibility does not by itself prove NORMAL.
- Temporal compatibility does not compensate for strong semantic
  incompatibility.
- Strong semantic compatibility must not cancel reliable evidence
  that the participant timelines are not normally coordinated.

============================================================
PARTICIPATION VALIDITY
============================================================

The field "speaks" is derived directly from the final filtered VAD turns:

- speaks = true means that the participant has at least one retained turn.
- speaks = false means that the participant has no retained turns during
  the entire 120-second interval.

Use the speaks fields.

Do not infer speaking activity from the semantic summaries.

For a normal spoken dyadic interaction, both participants are expected
to contribute speech during the complete interval.

Natural asymmetry is allowed:

- one participant may speak substantially more than the other,
- participants may have long listening periods,
- turn numbers and speaking durations do not need to be balanced.

However, the complete absence of retained speech from one participant
is not compatible with a normal two-person spoken interaction.

============================================================
LOCAL TEMPORAL FEATURES
============================================================

The signed strict A_end-to-B_start offsets are calculated as:

B_start minus A_end

Interpretation:

Negative value:

- Participant B starts shortly before Participant A finishes.

Value near zero:

- Participant B starts close to Participant A's turn boundary.

Positive value:

- Participant B starts after Participant A finishes.

The supplied local temporal evidence includes:

- the complete signed-offset list,
- number of valid offsets,
- mean,
- median,
- maximum,
- P75,
- P90,
- number of offsets above 1.5 seconds,
- percentage of offsets above 1.5 seconds.

Use the complete signed-offset distribution.

Do not decide from:

- one maximum value,
- one long pause,
- one negative value,
- or one isolated positive offset.

A NORMAL local temporal pattern is generally characterized by:

- repeated handoffs that are negative, near zero, or short positive,
- mean and median broadly compatible with the frozen NORMAL pattern,
- P75 and P90 broadly compatible with the frozen NORMAL pattern,
- offsets above 1.5 seconds being absent, uncommon, or isolated,
- no repeated and systematic pattern of substantially delayed handoffs.

Not every value must be negative or near zero.

A NORMAL interaction may contain occasional long pauses or unusual
handoffs.

However, repeated elevation across several parts of the offset
distribution is not equivalent to one isolated natural variation.

When the signed-offset list contains only one or two values, treat the
local temporal evidence as limited.

When the list is empty, the offset statistics are unavailable.
Do not invent missing evidence.

============================================================
FROZEN NORMAL LOCAL-TIMING REFERENCE
============================================================

The following statistics were calculated only from a frozen set of
separate NORMAL conversations:

Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

These statistics operationally describe the expected local temporal
pattern for NORMAL interactions in this experiment.

They are not independent hard thresholds.

A current case does not need to equal every average, and natural
variation around the reference pattern is expected.

However, these statistics are not optional background information.

Evaluate the current signed-offset distribution as a whole against the
frozen NORMAL reference.

A substantial and consistent departure across multiple reliable local
features means that the local temporal requirement for NORMAL is not
satisfied.

A multi-feature departure may include several of the following
appearing together:

- substantially elevated mean,
- substantially elevated median,
- substantially elevated P75 or P90,
- several offsets above 1.5 seconds,
- a substantially elevated percentage above 1.5 seconds,

One unusual feature alone is insufficient.

Several mutually supporting deviations are strong evidence.

============================================================
GLOBAL ALIGNMENT-SHIFT FEATURES
============================================================

The global alignment search evaluates hypothetical temporal corrections
to Participant B.

The global features have the following meanings:

- best_B_correction_shift_seconds is the hypothetical correction that
  produced the strongest bilateral turn-boundary alignment.

- A correction near zero means that little global temporal correction
  was preferred.

- A negative correction means that Participant B would align better if
  Participant B's timeline were moved earlier.

- estimated_B_lateness_seconds is the non-negative lateness implied by
  the best correction.

- alignment_score_gain_vs_zero measures how much the best correction
  improves alignment compared with applying no correction.

- best_num_bilateral_events indicates how many bilateral events support
  the selected correction.

- best_event_coverage indicates how much of the available interaction
  supports that estimate.

Do not use the correction value alone.

A large correction based on very few events or very low event coverage
is weak evidence.

A global estimate becomes more reliable when it is jointly supported by:

- a meaningful correction,
- meaningful estimated lateness,
- meaningful alignment improvement over zero shift,
- several bilateral events,
- sufficiently broad event coverage,
- and agreement with the local signed-offset pattern.

============================================================
FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE
============================================================

The following statistics were calculated only from the same frozen
NORMAL reference set:

Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.
- Median alignment score gain versus zero shift is around 0.009.
- Mean number of bilateral alignment events is around 11.94.
- Mean bilateral event coverage is around 59.52%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.

These statistics operationally describe the expected global alignment
behavior for NORMAL interactions in this experiment.

They are not independent hard thresholds.

Natural variation is allowed, and one unusual global value does not
automatically exclude NORMAL.

For NORMAL temporal coordination, the global evidence is generally
expected to show:

- a preferred correction near the frozen NORMAL pattern,
- limited estimated lateness,
- limited improvement over applying no correction,
- or insufficient reliable evidence that a substantial correction
  is necessary.

The global temporal requirement for NORMAL is not satisfied when a
substantial correction is reliably supported by the evidence as a whole.

Evaluate together:

- correction direction and magnitude,
- estimated lateness,
- alignment-score gain over zero shift,
- number of bilateral events,
- event coverage,
- agreement with the local offset distribution.

A substantial correction with negligible gain, very few events, or
very low coverage is weak evidence.

A substantial correction with meaningful gain, sufficient bilateral
events, broad coverage, and matching local evidence is strong evidence
that the interaction does not follow the frozen NORMAL temporal pattern.

============================================================
JOINT TEMPORAL COORDINATION DECISION
============================================================

Local and global temporal evidence must be evaluated together.

Temporal coordination is a necessary and independent property of a
NORMAL interaction.

A case can fail the temporal requirement for NORMAL even when:

- both participants speak,
- their semantic content is compatible,
- they discuss the same topic,
- they refer to the same people or events.

Semantic compatibility establishes content compatibility.
It does not establish temporal synchronization.

Treat the temporal dimension as compatible with NORMAL when:

- the signed-offset distribution is broadly consistent with the
  frozen NORMAL local pattern,
- long positive offsets are absent, uncommon, or isolated,
- elevated local values are not repeated across the distribution,
- the preferred global correction is near the frozen NORMAL pattern,
- or a larger correction is weakly supported by gain, events, or coverage.

Treat the temporal dimension as not compatible with NORMAL when:

- multiple local distribution statistics substantially depart from
  the frozen NORMAL pattern,
- delayed handoffs form a repeated rather than isolated pattern,
- and reliable global alignment evidence supports the same conclusion.

Do not require every temporal feature to depart from NORMAL.

Do not require every signed offset to be positive or large.

Do not allow one negative or near-zero offset to cancel a broader,
well-supported non-NORMAL temporal pattern.

When reliable local and global temporal evidence agree that the
interaction substantially departs from the frozen NORMAL pattern,
the complete case must be classified as ANOMALOUS, even when the
semantic evidence is fully compatible.

Do not infer or report the cause or magnitude of the temporal failure.

============================================================
SEMANTIC EVIDENCE
============================================================

The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and segment, you receive:

Coarse semantic information:

- speech_content_summary
- apparent_topic

Focused semantic information:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The semantic summaries were independently generated and may be broad,
imperfect, repetitive, or uncertain.

Evaluate semantic compatibility rather than exact wording.

A NORMAL semantic relationship may include:

- a shared concrete subject,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant supplying context for the other,
- one participant elaborating on or reacting to the other,
- compatible people, events, places, experiences, or arguments,
- different aspects of a shared subject,
- a coherent topic transition between Segment 0 and Segment 1.

Exact word matching and identical topic labels are not required.

A conversation may naturally change topic during the 120 seconds.

Broad labels such as:

- personal experiences,
- preferences,
- daily life,
- opinions,
- general discussion,
- lifestyle,
- personal well-being

are not sufficient evidence of semantic compatibility by themselves.

Concrete content must provide a plausible shared conversational context.

If one summary is vague, generic, unclear, or low-confidence, treat it
as limited evidence rather than automatically supporting either label.

Do not interpret the fact that both participants discuss generic
personal topics as sufficient evidence that they belong to one coherent
conversation.

Evaluate both synchronized segments and the complete 120-second semantic
relationship.

Do not use a simple vote between Segment 0 and Segment 1.

============================================================
FINAL COMBINED DECISION
============================================================

Internally evaluate:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Then make one binary decision.

Classify as NORMAL only when the complete evidence is sufficiently
compatible with all three required properties of a normal dyadic
interaction.

Classify as ANOMALOUS when at least one required property shows a strong,
reliable, and well-supported failure.

Do not require multiple properties to fail.

Do not allow strong evidence from one property to erase a reliable
failure in another property.

In particular:

- Semantic compatibility must not cancel reliable temporal failure.
- Temporal compatibility must not cancel strong semantic incompatibility.
- Speech from both participants must not cancel semantic or temporal failure.

Do not classify as ANOMALOUS because of one isolated noisy measurement.

Do not classify as NORMAL merely because one evidence source appears
plausible.

Use the reliability and consistency of the evidence, not a simple count
of supportive features.

Return one final label even when some evidence is limited.

============================================================
CURRENT CASE
============================================================

Analysis duration:
{duration_seconds} seconds

PARTICIPANT A

Speaks:
{participant_A_speaks}

PARTICIPANT B

Speaks:
{participant_B_speaks}

LOCAL TEMPORAL FEATURES

{local_temporal_features}

GLOBAL ALIGNMENT-SHIFT FEATURES

{global_shift_features}

SEMANTIC SUMMARIES

{semantic_summaries}

============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()

In [ ]:
# STEP L1.3 — PREPARE THE EXACT MANUALLY DEFINED EXPERIMENT

L1_NO_OVERLAP_CONFIG = prepare_manual_local_ablation_experiment(
    experiment_version=(
        "manual_ablation_l1_no_overlap_evidence_speaks_only"
    ),
    experiment_title=(
        "L1 — No Overlap Evidence — Manual Prompt — Speaks Only"
    ),
    ablation_id=(
        "L1_NO_OVERLAP_EVIDENCE_MANUAL_SPEAKS_ONLY"
    ),
    local_ablation_mode=(
        "NO_OVERLAP"
    ),
    manual_prompt_template=(
        L1_MANUAL_PROMPT_TEMPLATE
    ),
)


L1 — No Overlap Evidence — Manual Prompt — Speaks Only — MANUAL CONFIGURATION READY
Ablation ID: L1_NO_OVERLAP_EVIDENCE_MANUAL_SPEAKS_ONLY
Local ablation mode: NO_OVERLAP
Participant evidence: speaks only
Filtered turns passed to model: False
Automatic prompt rewriting used: False
Schema keys: ['participation_assessment', 'local_temporal_assessment', 'global_temporal_assessment', 'temporal_assessment', 'semantic_assessment', 'decisive_dimension', 'label']
Baseline prompt SHA256: cb64963d1881e6b5b14de4088dab4fcf97e95dbfa5c02bdc07051be896366b5c
Manual prompt SHA256: b6077b36aff2115441cfd76dbdec464758ee4c549233c92f4fe831aaca641d98
Prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/manual_prompt_diff_vs_speaks_only_baseline.txt


In [ ]:
# STEP L1.4 — REQUIRED PROMPT INSPECTION

L1_NO_OVERLAP_INSPECTION = inspect_manual_local_ablation_prompt(
    L1_NO_OVERLAP_CONFIG,
    case_index=0,
)


MANUAL PROMPT INSPECTION — L1 — No Overlap Evidence — Manual Prompt — Speaks Only
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.
Participant evidence supplied: speaks only.
Filtered turns supplied: False
Prompt definition method: manual complete template

EXACT ABLATED MODEL INPUT PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offsets_above_1_5_seconds": 2,
    "percent_offsets_above_1_5_seconds": 100.0
  },
  "global_shift_features": {
    "best_B_correction_shift_seconds": 2.7,
    "estimated_B_lateness_seconds": 0.0,
    "align

In [ ]:
# STEP L1.5 — RUN THE 400-CASE EXPERIMENT

L1_NO_OVERLAP_CACHE = run_manual_local_ablation_experiment(
    L1_NO_OVERLAP_CONFIG,
    print_each_case_prompt=False,
)


Resuming cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/predictions_cache.json
Existing records: 400


manual_ablation_l1_no_overlap_evidence_speaks_only:   0%|          | 0/400 [00:00<?, ?it/s]


L1 — No Overlap Evidence — Manual Prompt — Speaks Only — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/predictions_cache.json


In [ ]:
# STEP L1.6 — EVALUATE

L1_NO_OVERLAP_EVALUATION = evaluate_reasoning_experiment(
    L1_NO_OVERLAP_CONFIG
)


L1 — No Overlap Evidence — Manual Prompt — Speaks Only — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.9125
Balanced accuracy: 0.8617
ANOMALOUS precision: 0.9233
ANOMALOUS recall: 0.9633
ANOMALOUS F1: 0.9429
NORMAL recall / specificity: 0.7600
MCC: 0.7592
Matched source-group exact rate: 0.6700
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,76,24
Gold ANOMALOUS,11,289



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.873563,0.760000,0.812834,100.0000
ANOMALOUS,0.923323,0.963333,0.942904,300.0000
accuracy,0.912500,0.912500,0.912500,0.9125
macro avg,0.898443,0.861667,0.877869,400.0000
weighted avg,0.910883,0.912500,0.910386,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,6,44,44,0.88,0.88,1.0
1,lag_3sec,50,50,0,1,49,49,0.98,0.98,1.0
2,normal,100,100,0,76,24,76,0.76,0.76,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,4,96,96,0.96,0.96,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,NORMAL,165
3,local_temporal_assessment,LIMITED,134
4,local_temporal_assessment,ANOMALOUS,101
5,global_temporal_assessment,ANOMALOUS,169
6,global_temporal_assessment,LIMITED,134
7,global_temporal_assessment,NORMAL,97
8,temporal_assessment,ANOMALOUS,169
9,temporal_assessment,LIMITED,134



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/manual_prompt_diff_vs_speaks_only_baseline.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l1_no_overlap_evidence_speaks_only/classification_errors.csv

All 400 cases pr

In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD L1 — NO OVERLAP EVIDENCE
# ============================================================

if "L1_NO_OVERLAP_EVALUATION" in globals():

    l1_df = (
        L1_NO_OVERLAP_EVALUATION[
            "results_df"
        ].copy()
    )

elif "L1_NO_OVERLAP_CONFIG" in globals():

    l1_df = pd.read_csv(
        L1_NO_OVERLAP_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the L1_NO_OVERLAP configuration "
        "and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in l1_df.columns:

        l1_df[column] = (
            l1_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in l1_df.columns:

        l1_df[column] = (
            l1_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if "valid_prediction" not in l1_df.columns:

    l1_df["valid_prediction"] = (
        l1_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in l1_df.columns:

    l1_df["correct"] = (
        l1_df["valid_prediction"]
        &
        (
            l1_df["gold_label"]
            ==
            l1_df["prediction"]
        )
    )


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
# ============================================================

L1_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def l1_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(unexpected_values)
    )


    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_l1_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[row_field].fillna("MISSING"),
            subset[column_field].fillna("MISSING"),
            margins=True,
        )
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_l1_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset["gold_label"]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset["gold_label"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset["prediction"]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset["prediction"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset["valid_prediction"]
            ).sum()
        ),

        "correct": int(
            subset["correct"].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset["correct"].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    if "case_variant" in subset.columns:

        print(
            "\nCASE VARIANT BREAKDOWN"
        )


        variant_counts = (
            subset["case_variant"]
            .fillna("MISSING")
            .value_counts()
            .rename_axis("case_variant")
            .reset_index(name="count")
        )


        variant_counts["percentage"] = (
            100.0
            *
            variant_counts["count"]
            /
            len(subset)
        ).round(2)


        display(
            variant_counts
        )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            l1_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )
            for field, expected_values
            in L1_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_l1_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_l1_crosstab(
        subset,
        "local_temporal_assessment",
        "global_temporal_assessment",
        (
            "LOCAL × GLOBAL "
            "TEMPORAL ASSESSMENT"
        ),
    )


    display_l1_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "TEMPORAL ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_l1_crosstab(
        subset,
        "semantic_assessment",
        "decisive_dimension",
        (
            "SEMANTIC ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_l1_crosstab(
        subset,
        "temporal_assessment",
        "semantic_assessment",
        (
            "TEMPORAL × SEMANTIC ASSESSMENT"
        ),
    )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_l1_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe["case_family"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE OUTCOME SUBSET
# ============================================================

def get_l1_outcome_subset(
    family,
    outcome,
):

    family_mask = get_l1_family_mask(
        dataframe=l1_df,
        family=family,
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )


    return l1_df[
        family_mask
        &
        (
            l1_df["gold_label"]
            ==
            gold_label
        )
        &
        (
            l1_df["prediction"]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "L1 cases loaded:",
    len(l1_df),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    l1_df["case_family"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("case_family")
    .reset_index(name="count")
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        l1_df["case_family"],
        l1_df["prediction"],
        margins=True,
    )
)

L1 cases loaded: 400

CASE FAMILY COUNTS


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,93,7,100
normal,24,76,100
silent_partner,100,0,100
wrong_partner,96,4,100
All,313,87,400


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

l1_lag_correct = get_l1_outcome_subset(
    family="lag",
    outcome="correct",
)

inspect_l1_subset(
    l1_lag_correct,
    (
        "L1 — LAG CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

l1_lag_missed = get_l1_outcome_subset(
    family="lag",
    outcome="missed",
)

inspect_l1_subset(
    l1_lag_missed,
    (
        "L1 — LAG CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

l1_wrong_partner_correct = get_l1_outcome_subset(
    family="wrong_partner",
    outcome="correct",
)

inspect_l1_subset(
    l1_wrong_partner_correct,
    (
        "L1 — WRONG-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

l1_wrong_partner_missed = get_l1_outcome_subset(
    family="wrong_partner",
    outcome="missed",
)

inspect_l1_subset(
    l1_wrong_partner_missed,
    (
        "L1 — WRONG-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

l1_normal_correct = get_l1_outcome_subset(
    family="normal",
    outcome="correct",
)

inspect_l1_subset(
    l1_normal_correct,
    (
        "L1 — NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

l1_normal_missed = get_l1_outcome_subset(
    family="normal",
    outcome="missed",
)

inspect_l1_subset(
    l1_normal_missed,
    (
        "L1 — NORMAL CASES INCORRECTLY "
        "PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

l1_silent_partner_correct = get_l1_outcome_subset(
    family="silent_partner",
    outcome="correct",
)

inspect_l1_subset(
    l1_silent_partner_correct,
    (
        "L1 — SILENT-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

l1_silent_partner_missed = get_l1_outcome_subset(
    family="silent_partner",
    outcome="missed",
)

inspect_l1_subset(
    l1_silent_partner_missed,
    (
        "L1 — SILENT-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


L1 — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,93,0,93,0,93,0,93,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_3sec,49,52.69
1,lag_2sec,44,47.31



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,All
semantic_assessment,,
COMPATIBLE,19,19
LIMITED,81,81
All,100,100



TEMPORAL × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,LIMITED,All
temporal_assessment,,,
LIMITED,19,81,100
All,19,81,100



L1 — SILENT-PARTNER CASES NOT DETECTED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


# L2 — No Offset-Distribution Evidence

**Model-facing local evidence:** overlap only.

Removed as one conceptual group:

- `signed_strict_offsets_seconds`
- `num_signed_strict_offsets`
- `offset_mean_seconds`
- `offset_median_seconds`
- `offset_max_seconds`
- `offset_p75_seconds`
- `offset_p90_seconds`
- `num_offsets_above_1_5_seconds`
- `percent_offsets_above_1_5_seconds`

`local_temporal_assessment` remains in the structured output. Participant evidence is still `speaks` only.


In [ ]:
# STEP L2.1 — PRINT THE ORIGINAL SPEAKS-ONLY STRUCTURED R1 PROMPT
# Run only this cell first and send the printed prompt for manual adaptation.

L2_ORIGINAL_PROMPT = print_original_speaks_only_r1_prompt(
    experiment_title=(
        "L2 — No Offset-Distribution Evidence"
    ),
    case_index=0,
)


ORIGINAL STRUCTURED R1 PROMPT — SPEAKS ONLY
Target experiment: L2 — No Offset-Distribution Evidence
Inspection case ID: consolidation_lag_2sec_000
Filtered turns are absent from both the payload and prompt.
This is the unablated local-temporal baseline prompt that must be edited manually for the target experiment.

EXACT BASELINE MODEL-FACING PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "clean_overlap_seconds": 0.0,
    "clean_overlap_percent": 0.0,
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offsets_above_1_5_seconds": 2,
    "percent_offsets_above_1_5_seconds": 100.0
  },
  "global_shift_features": {
    "best_B_correction_shift_

In [ ]:
# STEP L2.2 — MANUAL PROMPT TEMPLATE
# Leave this as None until the complete L2 prompt has been manually defined.

L2_MANUAL_PROMPT_TEMPLATE = r"""
You are evaluating whether two participant records are jointly
consistent with one coherent, naturally synchronized, spoken
120-second dyadic interaction.

You are given information extracted independently for Participant A
and Participant B from the same 120-second analysis timeline.

Your task is binary classification only:

- NORMAL
- ANOMALOUS

Do not predict, infer, or name a specific anomaly type.

============================================================
AVAILABLE EVIDENCE
============================================================

You receive:

1. Whether each participant speaks during the 120-second interval.
2. Local filtered-overlap features.
3. Global temporal alignment-shift features.
4. Coarse semantic summaries for two synchronized 60-second segments.
5. Focused semantic summaries for the same two segments.
6. Frozen temporal reference statistics calculated only from
   separate NORMAL dyadic conversations.

Use only the supplied evidence.

Do not infer anything from identifiers, filenames, paths, dataset order,
sample position, participant identity, or hidden labels.

None of those fields are provided.

============================================================
OPERATIONAL DEFINITION OF NORMAL
============================================================

A NORMAL case must be compatible with one coherent, naturally
synchronized, two-person spoken interaction.

NORMAL requires three independent properties:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Evaluate these three properties separately before making the final
binary decision.

A case should be classified as NORMAL only when all three properties
are sufficiently supported by the available evidence.

A strong and reliable failure of any one property means that the
complete interaction does not satisfy the operational definition of
NORMAL.

Do not use a simple majority vote between the three properties.

Evidence that two properties appear normal must not override a strong
and reliable failure of the third property.

In particular:

- Both participants speaking does not by itself prove NORMAL.
- Semantic compatibility does not by itself prove NORMAL.
- Temporal compatibility does not compensate for strong semantic
  incompatibility.
- Strong semantic compatibility must not cancel reliable evidence
  that the participant timelines are not normally coordinated.

============================================================
PARTICIPATION VALIDITY
============================================================

The field "speaks" is derived directly from the final filtered VAD turns:

- speaks = true means that the participant has at least one retained turn.
- speaks = false means that the participant has no retained turns during
  the entire 120-second interval.

Use the speaks fields.

Do not infer speaking activity from the semantic summaries.

For a normal spoken dyadic interaction, both participants are expected
to contribute speech during the complete interval.

Natural asymmetry is allowed:

- one participant may speak substantially more than the other,
- participants may have long listening periods,
- turn numbers and speaking durations do not need to be balanced.

However, the complete absence of retained speech from one participant
is not compatible with a normal two-person spoken interaction.

============================================================
LOCAL TEMPORAL FEATURES
============================================================

The supplied local temporal evidence includes:

- filtered clean overlap in seconds,
- filtered clean overlap as a percentage of the complete interval.

Filtered overlap represents the amount of time during which both
participants have retained speech activity at the same time after the
independent backchannel-filtering procedure.

Use both the absolute overlap duration and the overlap percentage
together.

Do not decide from one overlap value in isolation from the complete
temporal and semantic evidence.

Natural overlap and interruption may occur in NORMAL interactions.

A current case does not need to equal the frozen NORMAL overlap pattern,
and natural variation around the reference pattern is expected.

When the overlap evidence is unusual but not supported by the remaining
evidence, treat the local temporal evidence cautiously.

Do not invent missing evidence.

============================================================
FROZEN NORMAL LOCAL-TIMING REFERENCE
============================================================

The following statistic was calculated only from a frozen set of
separate NORMAL conversations:

Frozen NORMAL local temporal reference statistic:

- Filtered clean overlap is around 6.25 seconds.

This value was calculated only from the frozen NORMAL reference conversations.
It is a soft reference pattern and must not be treated as a hard classification threshold.

This statistic operationally describes the expected local temporal
overlap pattern for NORMAL interactions in this experiment.

A current case does not need to equal the reference value, and natural
variation around the reference pattern is expected.

However, this statistic is not optional background information.

Evaluate the current filtered-overlap evidence against the frozen NORMAL
reference while respecting the limited reliability of a small local
feature group.

A substantial departure may contribute evidence that the local temporal
requirement for NORMAL is not satisfied.

One unusual overlap measurement alone is insufficient to determine the
complete interaction label.

============================================================
FILTERED OVERLAP
============================================================

Filtered overlap is the only supplied local temporal evidence in this
ablation.

Natural overlap and interruption may occur in NORMAL interactions.

Overlap alone must not determine the final label.

Interpret the overlap evidence together with:

- the global alignment evidence,
- the participation evidence,
- and the semantic evidence.

============================================================
GLOBAL ALIGNMENT-SHIFT FEATURES
============================================================

The global alignment search evaluates hypothetical temporal corrections
to Participant B.

The global features have the following meanings:

- best_B_correction_shift_seconds is the hypothetical correction that
  produced the strongest bilateral turn-boundary alignment.

- A correction near zero means that little global temporal correction
  was preferred.

- A negative correction means that Participant B would align better if
  Participant B's timeline were moved earlier.

- estimated_B_lateness_seconds is the non-negative lateness implied by
  the best correction.

- alignment_score_gain_vs_zero measures how much the best correction
  improves alignment compared with applying no correction.

- best_num_bilateral_events indicates how many bilateral events support
  the selected correction.

- best_event_coverage indicates how much of the available interaction
  supports that estimate.

Do not use the correction value alone.

A large correction based on very few events or very low event coverage
is weak evidence.

A global estimate becomes more reliable when it is jointly supported by:

- a meaningful correction,
- meaningful estimated lateness,
- meaningful alignment improvement over zero shift,
- several bilateral events,
- sufficiently broad event coverage,
- and agreement with the local filtered-overlap evidence.

============================================================
FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE
============================================================

The following statistics were calculated only from the same frozen
NORMAL reference set:

Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.
- Median alignment score gain versus zero shift is around 0.009.
- Mean number of bilateral alignment events is around 11.94.
- Mean bilateral event coverage is around 59.52%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.

These statistics operationally describe the expected global alignment
behavior for NORMAL interactions in this experiment.

They are not independent hard thresholds.

Natural variation is allowed, and one unusual global value does not
automatically exclude NORMAL.

For NORMAL temporal coordination, the global evidence is generally
expected to show:

- a preferred correction near the frozen NORMAL pattern,
- limited estimated lateness,
- limited improvement over applying no correction,
- or insufficient reliable evidence that a substantial correction
  is necessary.

The global temporal requirement for NORMAL is not satisfied when a
substantial correction is reliably supported by the evidence as a whole.

Evaluate together:

- correction direction and magnitude,
- estimated lateness,
- alignment-score gain over zero shift,
- number of bilateral events,
- event coverage,
- agreement with the local filtered-overlap evidence.

A substantial correction with negligible gain, very few events, or
very low coverage is weak evidence.

A substantial correction with meaningful gain, sufficient bilateral
events, broad coverage, and matching local evidence is strong evidence
that the interaction does not follow the frozen NORMAL temporal pattern.

============================================================
JOINT TEMPORAL COORDINATION DECISION
============================================================

Local and global temporal evidence must be evaluated together.

Temporal coordination is a necessary and independent property of a
NORMAL interaction.

A case can fail the temporal requirement for NORMAL even when:

- both participants speak,
- their semantic content is compatible,
- they discuss the same topic,
- they refer to the same people or events.

Semantic compatibility establishes content compatibility.
It does not establish temporal synchronization.

Treat the temporal dimension as compatible with NORMAL when:

- the filtered-overlap evidence is broadly consistent with the frozen
  NORMAL local pattern,
- an unusual overlap value is isolated or weakly supported,
- the preferred global correction is near the frozen NORMAL pattern,
- or a larger correction is weakly supported by gain, events, or coverage.

Treat the temporal dimension as not compatible with NORMAL when:

- the filtered-overlap evidence substantially departs from the frozen
  NORMAL pattern,
- and reliable global alignment evidence supports the same conclusion.

Do not require every temporal feature to depart from NORMAL.

Do not allow one apparently normal temporal value to cancel a broader,
well-supported non-NORMAL temporal pattern.

When reliable local and global temporal evidence agree that the
interaction substantially departs from the frozen NORMAL pattern,
the complete case must be classified as ANOMALOUS, even when the
semantic evidence is fully compatible.

Do not infer or report the cause or magnitude of the temporal failure.

============================================================
SEMANTIC EVIDENCE
============================================================

The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and segment, you receive:

Coarse semantic information:

- speech_content_summary
- apparent_topic

Focused semantic information:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The semantic summaries were independently generated and may be broad,
imperfect, repetitive, or uncertain.

Evaluate semantic compatibility rather than exact wording.

A NORMAL semantic relationship may include:

- a shared concrete subject,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant supplying context for the other,
- one participant elaborating on or reacting to the other,
- compatible people, events, places, experiences, or arguments,
- different aspects of a shared subject,
- a coherent topic transition between Segment 0 and Segment 1.

Exact word matching and identical topic labels are not required.

A conversation may naturally change topic during the 120 seconds.

Broad labels such as:

- personal experiences,
- preferences,
- daily life,
- opinions,
- general discussion,
- lifestyle,
- personal well-being

are not sufficient evidence of semantic compatibility by themselves.

Concrete content must provide a plausible shared conversational context.

If one summary is vague, generic, unclear, or low-confidence, treat it
as limited evidence rather than automatically supporting either label.

Do not interpret the fact that both participants discuss generic
personal topics as sufficient evidence that they belong to one coherent
conversation.

Evaluate both synchronized segments and the complete 120-second semantic
relationship.

Do not use a simple vote between Segment 0 and Segment 1.

============================================================
FINAL COMBINED DECISION
============================================================

Internally evaluate:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Then make one binary decision.

Classify as NORMAL only when the complete evidence is sufficiently
compatible with all three required properties of a normal dyadic
interaction.

Classify as ANOMALOUS when at least one required property shows a strong,
reliable, and well-supported failure.

Do not require multiple properties to fail.

Do not allow strong evidence from one property to erase a reliable
failure in another property.

In particular:

- Semantic compatibility must not cancel reliable temporal failure.
- Temporal compatibility must not cancel strong semantic incompatibility.
- Speech from both participants must not cancel semantic or temporal failure.

Do not classify as ANOMALOUS because of one isolated noisy measurement.

Do not classify as NORMAL merely because one evidence source appears
plausible.

Use the reliability and consistency of the evidence, not a simple count
of supportive features.

Return one final label even when some evidence is limited.

============================================================
CURRENT CASE
============================================================

Analysis duration:
{duration_seconds} seconds

PARTICIPANT A

Speaks:
{participant_A_speaks}

PARTICIPANT B

Speaks:
{participant_B_speaks}

LOCAL TEMPORAL FEATURES

{local_temporal_features}

GLOBAL ALIGNMENT-SHIFT FEATURES

{global_shift_features}

SEMANTIC SUMMARIES

{semantic_summaries}

============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()

In [ ]:
# STEP L2.3 — PREPARE THE EXACT MANUALLY DEFINED EXPERIMENT

L2_NO_OFFSET_DISTRIBUTION_CONFIG = prepare_manual_local_ablation_experiment(
    experiment_version=(
        "manual_ablation_l2_no_offset_distribution_speaks_only"
    ),
    experiment_title=(
        "L2 — No Offset-Distribution Evidence — Manual Prompt — Speaks Only"
    ),
    ablation_id=(
        "L2_NO_OFFSET_DISTRIBUTION_MANUAL_SPEAKS_ONLY"
    ),
    local_ablation_mode=(
        "NO_OFFSET_DISTRIBUTION"
    ),
    manual_prompt_template=(
        L2_MANUAL_PROMPT_TEMPLATE
    ),
)


L2 — No Offset-Distribution Evidence — Manual Prompt — Speaks Only — MANUAL CONFIGURATION READY
Ablation ID: L2_NO_OFFSET_DISTRIBUTION_MANUAL_SPEAKS_ONLY
Local ablation mode: NO_OFFSET_DISTRIBUTION
Participant evidence: speaks only
Filtered turns passed to model: False
Automatic prompt rewriting used: False
Schema keys: ['participation_assessment', 'local_temporal_assessment', 'global_temporal_assessment', 'temporal_assessment', 'semantic_assessment', 'decisive_dimension', 'label']
Baseline prompt SHA256: cb64963d1881e6b5b14de4088dab4fcf97e95dbfa5c02bdc07051be896366b5c
Manual prompt SHA256: 6ed5796e469bb2d1022d3c692e58ae0254849bbb91179c32705527b86d506e95
Prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/manual_prompt_diff_vs_speaks_only_baseline.txt


In [ ]:
# STEP L2.4 — REQUIRED PROMPT INSPECTION

L2_NO_OFFSET_DISTRIBUTION_INSPECTION = inspect_manual_local_ablation_prompt(
    L2_NO_OFFSET_DISTRIBUTION_CONFIG,
    case_index=0,
)


MANUAL PROMPT INSPECTION — L2 — No Offset-Distribution Evidence — Manual Prompt — Speaks Only
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.
Participant evidence supplied: speaks only.
Filtered turns supplied: False
Prompt definition method: manual complete template

EXACT ABLATED MODEL INPUT PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "clean_overlap_seconds": 0.0,
    "clean_overlap_percent": 0.0
  },
  "global_shift_features": {
    "best_B_correction_shift_seconds": 2.7,
    "estimated_B_lateness_seconds": 0.0,
    "alignment_score_gain_vs_zero": 0.069487,
    "best_num_bilateral_events": 5,
    "best_event_coverage_percent": 23.8095
  },
  "semantic_summaries": {
    "participant_A": {
      "segment_0": {
        "coarse_summary": {
          "speech_content_summary": "talking about a diffi

In [ ]:
# STEP L2.5 — RUN THE 400-CASE EXPERIMENT

L2_NO_OFFSET_DISTRIBUTION_CACHE = run_manual_local_ablation_experiment(
    L2_NO_OFFSET_DISTRIBUTION_CONFIG,
    print_each_case_prompt=False,
)


Resuming cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/predictions_cache.json
Existing records: 318


manual_ablation_l2_no_offset_distribution_speaks_only:   0%|          | 0/400 [00:00<?, ?it/s]


L2 — No Offset-Distribution Evidence — Manual Prompt — Speaks Only — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/predictions_cache.json


In [ ]:
# STEP L2.6 — EVALUATE

L2_NO_OFFSET_DISTRIBUTION_EVALUATION = evaluate_reasoning_experiment(
    L2_NO_OFFSET_DISTRIBUTION_CONFIG
)


L2 — No Offset-Distribution Evidence — Manual Prompt — Speaks Only — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8275
Balanced accuracy: 0.6883
ANOMALOUS precision: 0.8309
ANOMALOUS recall: 0.9667
ANOMALOUS F1: 0.8937
NORMAL recall / specificity: 0.4100
MCC: 0.4890
Matched source-group exact rate: 0.3800
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,41,59
Gold ANOMALOUS,10,290



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.803922,0.410000,0.543046,100.0000
ANOMALOUS,0.830946,0.966667,0.893683,300.0000
accuracy,0.827500,0.827500,0.827500,0.8275
macro avg,0.817434,0.688333,0.718364,400.0000
weighted avg,0.824190,0.827500,0.806024,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,4,46,46,0.92,0.92,1.0
1,lag_3sec,50,50,0,1,49,49,0.98,0.98,1.0
2,normal,100,100,0,41,59,41,0.41,0.41,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,5,95,95,0.95,0.95,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,ANOMALOUS,176
3,local_temporal_assessment,LIMITED,117
4,local_temporal_assessment,NORMAL,107
5,global_temporal_assessment,ANOMALOUS,226
6,global_temporal_assessment,LIMITED,117
7,global_temporal_assessment,NORMAL,57
8,temporal_assessment,ANOMALOUS,226
9,temporal_assessment,LIMITED,117



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/manual_prompt_diff_vs_speaks_only_baseline.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_l2_no_offset_distribution_speaks_only/classification_errors.csv

In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD L2 — NO OFFSET-DISTRIBUTION EVIDENCE
# ============================================================

if "L2_NO_OFFSET_DISTRIBUTION_EVALUATION" in globals():

    l2_df = (
        L2_NO_OFFSET_DISTRIBUTION_EVALUATION[
            "results_df"
        ].copy()
    )

elif "L2_NO_OFFSET_DISTRIBUTION_CONFIG" in globals():

    l2_df = pd.read_csv(
        L2_NO_OFFSET_DISTRIBUTION_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the L2_NO_OFFSET_DISTRIBUTION "
        "configuration and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in l2_df.columns:

        l2_df[column] = (
            l2_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in l2_df.columns:

        l2_df[column] = (
            l2_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if "valid_prediction" not in l2_df.columns:

    l2_df["valid_prediction"] = (
        l2_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in l2_df.columns:

    l2_df["correct"] = (
        l2_df["valid_prediction"]
        &
        (
            l2_df["gold_label"]
            ==
            l2_df["prediction"]
        )
    )


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
# ============================================================

L2_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def l2_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(unexpected_values)
    )


    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_l2_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[row_field].fillna("MISSING"),
            subset[column_field].fillna("MISSING"),
            margins=True,
        )
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_l2_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset["gold_label"]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset["gold_label"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset["prediction"]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset["prediction"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset["valid_prediction"]
            ).sum()
        ),

        "correct": int(
            subset["correct"].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset["correct"].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    if "case_variant" in subset.columns:

        print(
            "\nCASE VARIANT BREAKDOWN"
        )


        variant_counts = (
            subset["case_variant"]
            .fillna("MISSING")
            .value_counts()
            .rename_axis("case_variant")
            .reset_index(name="count")
        )


        variant_counts["percentage"] = (
            100.0
            *
            variant_counts["count"]
            /
            len(subset)
        ).round(2)


        display(
            variant_counts
        )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            l2_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )
            for field, expected_values
            in L2_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_l2_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_l2_crosstab(
        subset,
        "local_temporal_assessment",
        "global_temporal_assessment",
        (
            "LOCAL × GLOBAL "
            "TEMPORAL ASSESSMENT"
        ),
    )


    display_l2_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "TEMPORAL ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_l2_crosstab(
        subset,
        "semantic_assessment",
        "decisive_dimension",
        (
            "SEMANTIC ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_l2_crosstab(
        subset,
        "temporal_assessment",
        "semantic_assessment",
        (
            "TEMPORAL × SEMANTIC ASSESSMENT"
        ),
    )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_l2_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe["case_family"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE OUTCOME SUBSET
# ============================================================

def get_l2_outcome_subset(
    family,
    outcome,
):

    family_mask = get_l2_family_mask(
        dataframe=l2_df,
        family=family,
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )


    return l2_df[
        family_mask
        &
        (
            l2_df["gold_label"]
            ==
            gold_label
        )
        &
        (
            l2_df["prediction"]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "L2 cases loaded:",
    len(l2_df),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    l2_df["case_family"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("case_family")
    .reset_index(name="count")
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        l2_df["case_family"],
        l2_df["prediction"],
        margins=True,
    )
)

L2 cases loaded: 400

CASE FAMILY COUNTS


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,95,5,100
normal,59,41,100
silent_partner,100,0,100
wrong_partner,95,5,100
All,349,51,400


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

l2_lag_correct = get_l2_outcome_subset(
    family="lag",
    outcome="correct",
)

inspect_l2_subset(
    l2_lag_correct,
    (
        "L2 — LAG CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

l2_lag_missed = get_l2_outcome_subset(
    family="lag",
    outcome="missed",
)

inspect_l2_subset(
    l2_lag_missed,
    (
        "L2 — LAG CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

l2_wrong_partner_correct = get_l2_outcome_subset(
    family="wrong_partner",
    outcome="correct",
)

inspect_l2_subset(
    l2_wrong_partner_correct,
    (
        "L2 — WRONG-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

l2_wrong_partner_missed = get_l2_outcome_subset(
    family="wrong_partner",
    outcome="missed",
)

inspect_l2_subset(
    l2_wrong_partner_missed,
    (
        "L2 — WRONG-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

l2_normal_correct = get_l2_outcome_subset(
    family="normal",
    outcome="correct",
)

inspect_l2_subset(
    l2_normal_correct,
    (
        "L2 — NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

l2_normal_missed = get_l2_outcome_subset(
    family="normal",
    outcome="missed",
)

inspect_l2_subset(
    l2_normal_missed,
    (
        "L2 — NORMAL CASES INCORRECTLY "
        "PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

l2_silent_partner_correct = get_l2_outcome_subset(
    family="silent_partner",
    outcome="correct",
)

inspect_l2_subset(
    l2_silent_partner_correct,
    (
        "L2 — SILENT-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

l2_silent_partner_missed = get_l2_outcome_subset(
    family="silent_partner",
    outcome="missed",
)

inspect_l2_subset(
    l2_silent_partner_missed,
    (
        "L2 — SILENT-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


L2 — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,95,0,95,0,95,0,95,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_3sec,49,51.58
1,lag_2sec,46,48.42



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,TEMPORAL,All
semantic_assessment,,,
COMPATIBLE,34,13,47
LIMITED,53,0,53
All,87,13,100



TEMPORAL × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,LIMITED,All
temporal_assessment,,,
ANOMALOUS,44,48,92
LIMITED,3,5,8
All,47,53,100



L2 — SILENT-PARTNER CASES NOT DETECTED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


# Compare available experiments

This cell safely includes only experiments whose evaluation has already been completed or whose metrics file exists.


In [ ]:
# ============================================================
# COMPARE AVAILABLE MANUAL LOCAL ABLATIONS
# ============================================================

comparison_rows = []


def append_metrics_row(
    experiment_name,
    metrics,
):
    comparison_rows.append({
        "experiment": experiment_name,
        "accuracy": metrics.get(
            "accuracy_valid_predictions"
        ),
        "balanced_accuracy": metrics.get(
            "balanced_accuracy"
        ),
        "anomalous_precision": metrics.get(
            "anomalous_precision"
        ),
        "anomalous_recall": metrics.get(
            "anomalous_recall"
        ),
        "anomalous_f1": metrics.get(
            "anomalous_f1"
        ),
        "normal_recall_specificity": metrics.get(
            "normal_recall_specificity"
        ),
        "mcc": metrics.get(
            "matthews_correlation_coefficient"
        ),
        "exact_schema_rate": metrics.get(
            "exact_json_schema_rate"
        ),
        "source_group_exact_match_rate": metrics.get(
            "source_group_exact_match_rate"
        ),
    })


for experiment_name, evaluation_name, config_name in [
    (
        "A2 — No Local Branch — Manual — Speaks Only",
        "A2_NO_LOCAL_EVALUATION",
        "A2_NO_LOCAL_CONFIG",
    ),
    (
        "L1 — No Overlap — Manual — Speaks Only",
        "L1_NO_OVERLAP_EVALUATION",
        "L1_NO_OVERLAP_CONFIG",
    ),
    (
        "L2 — No Offset Distribution — Manual — Speaks Only",
        "L2_NO_OFFSET_DISTRIBUTION_EVALUATION",
        "L2_NO_OFFSET_DISTRIBUTION_CONFIG",
    ),
]:
    evaluation = globals().get(
        evaluation_name
    )
    config = globals().get(
        config_name
    )

    if evaluation is not None:
        append_metrics_row(
            experiment_name,
            evaluation[
                "metrics"
            ],
        )

    elif (
        config is not None
        and config[
            "paths"
        ][
            "metrics_json"
        ].exists()
    ):
        append_metrics_row(
            experiment_name,
            json.loads(
                config[
                    "paths"
                ][
                    "metrics_json"
                ].read_text(
                    encoding="utf-8"
                )
            ),
        )


manual_local_ablation_comparison_df = pd.DataFrame(
    comparison_rows
)

display(
    manual_local_ablation_comparison_df
)


# Optional: disconnect the Colab runtime


In [ ]:
from google.colab import runtime

runtime.unassign()
